# Milestone 1 — Thu thập dữ liệu arXiv

**Môn học:** CSC14119 — Nhập môn Khoa học Dữ liệu  ·  **Nhóm 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuyNhat1705/Intro2DS-Group3-Source/blob/main/src/scraper.ipynb)

Notebook này là chương trình thu thập dữ liệu của nhóm cho Milestone 1. Với mỗi arXiv ID được giao, scraper lấy **metadata**, **full-text LaTeX source của mọi version** (chỉ giữ `.tex` / `.bib`) và **danh sách reference** từ Semantic Scholar. Trong lúc chạy, notebook đo thời gian, RAM, dung lượng đĩa, rồi xuất số liệu cho báo cáo Jupyter Book.

> **Chạy nhanh:** chọn range ở ô ngay dưới, rồi *Runtime → Run all*. Hướng dẫn đầy đủ ở **mục 2**.

In [ ]:
# CHỌN RANGE — sửa ô này trước khi Run all (xem mục 2)
RANGES = {                      # số range -> (YYMM, số đầu, số cuối), inclusive; giống nhau trên máy của mọi người
    1: ("2609", 27834, 32833),
    2: ("2609", 17834, 22833),
    3: ("2608", 14007, 19006),
    4: ("2608", 4007, 9006),
    5: ("2511", 6709, 11708),
}
RUN_RANGES = [1]                # range máy này chạy: [3] = range 3; [1, 2] = range 1 và 2; "all" = cả 5 range
SAMPLE_N = None                 # chạy thử: vd 20 = 20 bài đầu mỗi range; None = toàn bộ

## Mục lục

1. Giới thiệu
2. Hướng dẫn cài đặt và chạy
3. Thiết kế scraper
4. Thiết lập môi trường
5. Cấu hình
6. Lưu trữ dữ liệu và module
7. Các thành phần hỗ trợ
8. Thu thập dữ liệu
9. Kết quả: dữ liệu thu được
10. Kết quả: hiệu năng
11. Xuất số liệu cho Jupyter Book
12. Kết luận

Mục 1–3 là phần mô tả, không có code. Mục 4–8 cài đặt môi trường và cào dữ liệu. Mục 9–12 chỉ đọc dữ liệu đã có trên đĩa để thống kê.

## 1. Giới thiệu

### 1.1 Mục tiêu

Theo yêu cầu của Milestone 1, với mỗi bài trong dải arXiv ID được giao, nhóm cần:

1. **Entry discovery**: phân giải ID thành metadata (tiêu đề, tác giả, ngày nộp, các ngày sửa, venue) và biết bài có bao nhiêu version.
2. **Full-text source**: tải source của **từng version** qua endpoint có version `export.arxiv.org/src/<id>v<n>`. Chỉ giữ file `.tex` và `.bib` (bỏ hình bằng cách extract chọn lọc). Version chỉ có PDF hoặc định dạng khác vẫn có một bản ghi rỗng.
3. **References**: lấy danh sách reference của mỗi bài qua Semantic Scholar API. Giữ lại mọi reference ở local; reference có arXiv ID trở thành một dòng của bảng `references`.
4. **Báo cáo hiệu năng**: thời gian xử lý mỗi bài, tổng thời gian entry discovery, RAM tối đa / trung bình, dung lượng đĩa tối đa, dung lượng output cuối. Testbed là Google Colab, chế độ CPU.

### 1.2 Phạm vi dữ liệu

Nhóm được giao 25.000 ID, chia thành 5 range liên tiếp. Số version theo lần discovery ngày 2026-10-26:

| Range | arXiv ID | Số bài | Bài chỉ có v1 | Tổng số version |
|---|---|---|---|---|
| 1 | `2609.27834` – `2609.32833` | 5.000 | 87,6% | 5.702 |
| 2 | `2609.17834` – `2609.22833` | 5.000 | 85,3% | 5.843 |
| 3 | `2608.14007` – `2608.19006` | 5.000 | 81,6% | 6.097 |
| 4 | `2608.04007` – `2608.09006` | 5.000 | 82,1% | 6.088 |
| 5 | `2511.06709` – `2511.11708` | 5.000 | 60,8% | 7.800 |
| **Tổng** | | **25.000** | **79,5%** | **31.530** |

### 1.3 Đầu ra

- Mỗi bài một thư mục `data/<yymm-nnnnn>/` gồm `metadata.json`, `references.json` và `v1/`, `v2/`, … Mỗi thư mục version chỉ chứa `.tex` / `.bib`, giữ nguyên cấu trúc thư mục con của source.
- Hai bảng theo đề, lưu dạng Parquet trong `data/tables/`:

| Bảng | Cột |
|---|---|
| `papers` | `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue` |
| `references` | `arxiv_id`, `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_s2_id` |

- Số liệu thống kê và hiệu năng trong `data/report_exports/`, để Jupyter Book vẽ hình lúc build.

## 2. Hướng dẫn cài đặt và chạy

### 2.1 Yêu cầu

- Python **3.12** (pandas 3.x cần Python ≥ 3.11). Thư viện và phiên bản nằm trong `src/requirements.txt`; môi trường conda tương ứng là `src/intro2ds-env.yml`.
- Tài khoản Google, để dùng Colab và lưu dữ liệu trên Google Drive (corpus `.tex` / `.bib` cỡ vài GB).
- Khuyến nghị: **API key Semantic Scholar** (miễn phí). Không có key vẫn chạy được, nhưng dùng pool chung nên dễ bị 429 và chậm hơn.
- arXiv API, `export.arxiv.org` và Google Drive không cần key.

### 2.2 Chạy trên Google Colab (testbed của đề)

1. Mở notebook bằng nút *Open in Colab* ở đầu trang. Chọn *Runtime → Change runtime type* → **CPU**.
2. Thêm API key Semantic Scholar vào *Secrets* (biểu tượng chìa khoá) với tên `S2_API_KEY` và bật *Notebook access*. Có key thì notebook tự giảm nhịp Semantic Scholar từ 3 giây xuống `S2_DELAY_WITH_KEY` (1 giây); không có key thì giữ 3 giây.
3. Ở ô **Chọn range** đầu notebook, đặt `RUN_RANGES` là range mình được giao, ví dụ `[3]`. Mỗi range nên chạy trên một Colab riêng; `"all"` là một máy chạy cả 5 range (khoảng 26 giờ, 3 phiên).
4. *Runtime → Run all*:
   - Mục 4 clone repo (hoặc `git pull` nếu runtime đã có repo) và cài thư viện theo `src/requirements.txt`. Nếu notebook báo cần restart thì chọn *Runtime → Restart session* rồi *Run all* lại.
   - Mục 6 mount Google Drive; `data/` nằm ở `DRIVE_DATA_DIR` (mặc định `MyDrive/Intro2DS-Group3/data`).
   - Mục 8 cào dữ liệu tới khi xong, khi đạt `MAX_RUN_HOURS` (11,5 giờ, trước khi Colab cắt phiên) hoặc khi dịch vụ từ chối. Sau đó mục 9–11 cập nhật bảng và số liệu.
5. Giữ tab trình duyệt mở trong lúc chạy: Colab miễn phí ngắt phiên khi không có tương tác.

### 2.3 Chạy trên máy cá nhân

```bash
pip install -r src/requirements.txt          # hoặc: conda env create -f src/intro2ds-env.yml
jupyter notebook src/scraper.ipynb
```

Mở notebook từ bên trong repo; dữ liệu nằm ở `<repo>/data`. Đặt key bằng biến môi trường `S2_API_KEY` trước khi mở Jupyter.

### 2.4 Chạy thử, dừng và chạy tiếp

- **Chạy thử**: đặt `SAMPLE_N = 20` ở ô Chọn range để chỉ cào 20 bài đầu mỗi range. Dữ liệu chạy thử nằm cùng chỗ với dữ liệu thật, nên lần chạy thật dùng lại được.
- **Dừng**: bấm *Interrupt*. Notebook chờ tối đa `INTERRUPT_GRACE` giây cho các làn xong việc hiện tại, rồi ghi state và log.
- **Chạy tiếp**: mở lại notebook và *Run all*. Tiến độ được dựng lại từ các file state trên đĩa, việc đã xong không làm lại. Lặp lại tới khi mục 8.1 báo không còn việc. Nếu mất phiên đột ngột, chỉ mất tối đa `SOURCE_STATE_FLUSH_EVERY` giây tiến độ; các version đó được tải lại.

### 2.5 Gộp dữ liệu và báo cáo cả nhóm

- **Lưu chung**: một người tạo thư mục `Intro2DS-Group3` trên Drive và chia sẻ quyền *Editor* cho cả nhóm; mỗi người *Add shortcut to My Drive*. State và log của mỗi máy nằm trong thư mục con riêng đặt theo range (`data/.state/R3/`, `data/run_logs/R3/`), và thư mục bài không trùng nhau giữa các range, nên không ai ghi đè file của người khác. Nếu mỗi người dùng Drive riêng thì cuối cùng chép các thư mục `data/` vào chung một chỗ.
- **Báo cáo cả nhóm**: khi `data/` đã có đủ dữ liệu, đặt `REPORT_ALL_RANGES = True` ở mục 5 rồi *Run all*. Mục 8 báo không còn việc; mục 9–11 tính trên cả 5 range và log của mọi máy.

## 3. Thiết kế scraper

### 3.1 Nguồn dữ liệu và công cụ

| Bước | Nguồn | Công cụ | Module |
|---|---|---|---|
| Entry discovery | arXiv API (`export.arxiv.org/api/query`, `id_list` tới 100 ID mỗi request) | `arxiv` 4.x | `get_metadata.py` |
| Full-text source | `export.arxiv.org/src/<id>v<n>` (từng version) | `requests`, `tarfile`, `gzip`, `zipfile` | `get_full_source.py` |
| References | Semantic Scholar Graph API (`/paper/batch`, `/paper/arXiv:<id>/references`) | `requests` | `get_extracting_reference.py` |
| Đo hiệu năng | tiến trình Python, thư mục `data/` | `psutil`, `resource` | notebook (mục 7.2) |

Notebook chỉ **gọi** các module trong `src/`. Bản hybrid đọc source từ bucket S3 của arXiv nằm riêng ở `src/ExportS3/`, chưa dùng ở đây.

### 3.2 Kiến trúc

Quy tắc của arXiv cho mỗi client là **một kết nối tại một thời điểm, ≥ 3 giây giữa hai request**. 31.530 version tải lần lượt thì mất khoảng 26 giờ trên một máy. Vì vậy scraper song song ở hai mức:

- **Theo máy**: mỗi range chạy trên một instance Colab riêng. Mỗi máy vẫn giữ đúng một kết nối tới arXiv. Một range cần khoảng 5–6,5 giờ, vừa trong một phiên Colab.
- **Theo dịch vụ, trong một máy**: các làn chạy song song trong cùng một tiến trình:

```
            ┌──── Thread "arxiv" (1 kết nối, ≥ ARXIV_DELAY giây giữa lúc BẮT ĐẦU hai request) ────┐
 ID list ──►│ Phase A: metadata theo batch cho mọi bài ──► Phase B: /src/ v1..vN từng bài          │
            └──────────────────────────────────────────────┬────────────────────────────────────────┘
            ┌──── Thread "source-writer" ──────────────────▼────────────────────────────────────────┐
            │ giải nén .tex/.bib từ RAM, ghi ra data/<id>/v<n>/, cập nhật state                      │
            └───────────────────────────────────────────────────────────────────────────────────────┘
            ┌──── Thread "s2" (độc lập) ────────────────────────────────────────────────────────────┐
 ID list ──►│ Phase C: references theo batch (references.json)                                      │
            └───────────────────────────────────────────────────────────────────────────────────────┘
            ┌──── Thread "monitor" ─────────────────────────────────────────────────────────────────┐
            │ RSS tiến trình (psutil) mỗi MONITOR_EVERY giây, dung lượng data/ mỗi DISK_EVERY       │
            └───────────────────────────────────────────────────────────────────────────────────────┘
```

### 3.3 Tối ưu đường găng

Đường găng là Phase B, vì mỗi version cần một request `/src/`. Các tối ưu:

- **Nhịp tính từ lúc request bắt đầu** (`gfs.Pacer`): thời gian tải nằm trong 3 giây chứ không cộng thêm. Một nhịp dùng chung cho API metadata và `/src/`.
- **Metadata theo batch**: 100 bài mỗi request, nên Phase A của một range chỉ khoảng 60 request (vài phút).
- **Bỏ qua body của version chỉ có PDF**: nhận ra qua `Content-Type`, không tải file PDF (thường vài MB).
- **Giải nén trong RAM, một lượt**: chỉ ghi `.tex` / `.bib`; cùng lượt đó đo kích thước cả archive (tức "trước khi bỏ hình"). Archive và hình không bao giờ ghi ra đĩa.
- **Ghi ở thread nền** (`gfs.Writer`): request kế tiếp bắt đầu đúng nhịp, không chờ ghi xong archive trước.
- **State trong RAM**: `source_state.json` ghi xuống Drive mỗi `SOURCE_STATE_FLUSH_EVERY` giây thay vì sau mỗi version.

### 3.4 Xử lý lỗi và chạy tiếp

- **429 / 5xx**: module chờ (30, 60, 120 giây) rồi thử lại, đồng thời tự giãn nhịp thêm 1 giây mỗi lần (tối đa `ARXIV_MAX_DELAY`). Sau 100 request thành công liên tiếp, nhịp giảm dần về `ARXIV_DELAY`.
- **Tải bị đứt giữa chừng**: thử lại sau 5, 10, 15 giây.
- Mọi lỗi của module được quy về ba loại:

| Loại | Ý nghĩa | Xử lý |
|---|---|---|
| `PaperError(status)` | lỗi riêng của một bài (`not_found`, `failed`) | ghi lại rồi đi tiếp; `failed` được thử lại ở phiên sau |
| `ServiceStop` | dịch vụ từ chối (403, hết quota Semantic Scholar) | dừng **làn đó**, làn còn lại chạy tiếp |
| `ServiceStop(retryable=True)` | arXiv vẫn trả 429 / 5xx sau khi module tự chờ | nghỉ theo `COOLDOWNS_SEC` (5, 10, 15 phút) rồi thử lại **cùng request**; hết danh sách mới dừng làn |

- **Chạy tiếp**: tiến độ luôn được dựng lại từ các file state trên đĩa (mục 7.1). Bị dừng vì bất kỳ lý do nào (hết giờ Colab, 403, Interrupt), chỉ cần chạy lại notebook.

### 3.5 Tổ chức dữ liệu

```
data/
├── <yymm-nnnnn>/            # mỗi bài một thư mục (vd 2608-04007)
│   ├── metadata.json        # get_metadata
│   ├── references.json      # get_extracting_reference (mọi reference)
│   └── v1/, v2/, ...        # get_full_source: chỉ .tex/.bib (rỗng nếu version chỉ có PDF)
├── .state/<WORKER>/         # source_state.json, refs_done.jsonl, nb_status.jsonl
├── run_logs/<WORKER>/       # runs.jsonl, timings.jsonl, events.jsonl, monitor_*.csv, disk_*.csv, scraper.log
├── tables/                  # papers.parquet, references.parquet, references_all.parquet
└── report_exports/          # số liệu nhỏ cho Jupyter Book
```

`WORKER` là tên máy, tự sinh từ range (`R3`, `R1+R2`, `all`). File state nằm thẳng trong `data/` hoặc `data/.state/` là của bản trước khi chia theo range; notebook vẫn đọc chúng để biết bài nào đã xong.

## 4. Thiết lập môi trường

- **4.1** Tìm thư mục repo (trên Colab: clone nhánh `BRANCH`, hoặc `git pull` nếu đã clone), rồi thêm `src/` vào `sys.path`.
- **4.2** Kiểm tra phiên bản thư viện theo `src/requirements.txt`; trên Colab, notebook tự cài.
- **4.3** Ghi thông tin testbed (CPU, RAM, đĩa, phiên bản thư viện). Thông tin này được lưu vào log của mỗi phiên.

In [ ]:
# 4.1 — Tìm repo (Colab: clone), thêm src/ vào sys.path
import os
import sys
import json
import math
import re
import time
import shutil
import platform
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH = "scraper"  # nhánh chứa code scraper; đổi thành "main" sau khi merge
COLAB_REPO_DIR = Path("/content/Intro2DS-Group3-Source")


def find_repo_root(start: Path):
    """Đi ngược lên từ `start` tới thư mục chứa src/utils.py."""
    for p in (start, *start.parents):
        if (p / "src" / "utils.py").is_file():
            return p
    return None


if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(COLAB_REPO_DIR)],
                       check=True)
    else:  # runtime đã clone từ trước: lấy code mới nhất (ô này chạy trước khi import module)
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull", "--ff-only"], check=False)
    REPO_ROOT = COLAB_REPO_DIR
else:
    REPO_ROOT = find_repo_root(Path.cwd().resolve())
    if REPO_ROOT is None:
        raise RuntimeError("Không tìm thấy repo (src/utils.py). Hãy mở notebook từ bên trong repo.")

SRC_DIR = REPO_ROOT / "src"
# Các module import utils theo nhiều kiểu (from src.utils / from utils) nên thêm cả gốc repo lẫn src/
for p in (str(SRC_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

print("Colab:", IN_COLAB)
print("Repo :", REPO_ROOT)

In [ ]:
# 4.2 — Kiểm tra thư viện theo src/requirements.txt (Colab: tự cài)
from importlib import metadata as importlib_metadata

REQUIREMENTS_FILE = SRC_DIR / "requirements.txt"


def read_requirements(path: Path) -> dict:
    """{tên: phiên bản} từ các dòng `tên==phiên_bản`."""
    reqs = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if "==" in line:
            name, version = line.split("==", 1)
            reqs[name.strip()] = version.strip()
    return reqs


def installed_version(name: str):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


REQS = read_requirements(REQUIREMENTS_FILE)
mismatches = {n: (installed_version(n), v) for n, v in REQS.items() if installed_version(n) != v}

if not mismatches:
    print("Thư viện đúng phiên bản trong requirements.txt.")
elif IN_COLAB:
    already_loaded = [n for n in mismatches if n.replace("-", "_") in sys.modules]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_FILE)], check=True)
    if already_loaded:
        raise RuntimeError(
            f"Đã cài lại {already_loaded} (đang được import sẵn trong kernel). "
            "Vào Runtime > Restart session rồi chạy lại notebook từ đầu."
        )
    print("Đã cài:", {n: v for n, (_, v) in mismatches.items()})
else:
    print("CẢNH BÁO — khác requirements.txt {tên: (đang có, cần)}:", mismatches)
    print(f"Cài bằng: {sys.executable} -m pip install -r {REQUIREMENTS_FILE}")

In [ ]:
# 4.3 — Thông tin testbed (ghi vào runs.jsonl ở mỗi phiên)
import psutil


def cpu_model() -> str:
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def machine_info() -> dict:
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage(REPO_ROOT)
    return {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "colab": IN_COLAB,
        "cpu_model": cpu_model(),
        "cpu_count": os.cpu_count(),
        "ram_total_bytes": vm.total,
        "disk_total_bytes": disk.total,
        "disk_free_bytes": disk.free,
        "packages": {n: installed_version(n) for n in REQS},
    }


print(json.dumps(machine_info(), indent=2, ensure_ascii=False))

## 5. Cấu hình

Range máy này chạy nằm ở ô *Chọn range* đầu notebook. Các tham số còn lại nằm trong ô 5.1; ô 5.2 kiểm tra chúng và nạp key.

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `ARXIV_DELAY` | 3,0 | giây giữa lúc bắt đầu hai request arXiv (đề yêu cầu ≥ 3) |
| `ARXIV_MAX_DELAY` | 6,0 | nhịp tối đa khi tự giãn sau 429 / 5xx |
| `COOLDOWNS_SEC` | 300, 600, 900 | các lần nghỉ khi arXiv vẫn trả 429 sau khi module tự chờ |
| `METADATA_BATCH_SIZE` | 100 | số bài mỗi request metadata |
| `S2_BATCH_SIZE`, `S2_DELAY_WITH_KEY` | 100; 1,0 | số bài mỗi request Semantic Scholar; nhịp khi có key |
| `SOURCE_WRITE_QUEUE` | 3 | số version đã tải chờ ghi ở thread nền (0 = ghi ngay) |
| `SOURCE_STATE_FLUSH_EVERY` | 120 | giây giữa hai lần ghi state source xuống đĩa |
| `MAX_RUN_HOURS` | 11,5 | tự dừng gọn trước khi Colab cắt phiên |
| `REPORT_ALL_RANGES` | `False` | mục 9–11 tính cho range của máy này, hay cho cả 5 range |
| `PERSIST_TO_DRIVE`, `DRIVE_DATA_DIR` | `True` | lưu `data/` trên Google Drive (Colab) |

API key **không** ghi vào notebook. Đặt biến môi trường `S2_API_KEY`, hoặc trên Colab thêm vào *Secrets* với tên `S2_API_KEY`.

In [ ]:
# 5.1 — Tham số
RUN_METADATA = True             # Phase A
RUN_SOURCE = True               # Phase B
RUN_REFS = True                 # Phase C
PARALLEL = True                 # True: làn arXiv ‖ làn S2; False: chạy tuần tự (để so sánh)
REPORT_ALL_RANGES = False       # mục 9–11: False = các range của máy này; True = cả 5 range (khi data/ đã có dữ liệu cả nhóm)

ARXIV_DELAY = 3.0               # giây giữa lúc BẮT ĐẦU hai request arXiv liên tiếp (đề: >= 3s)
ARXIV_MAX_DELAY = 6.0           # gặp 429/5xx: nhịp giãn dần tới mức này, ổn định lại thì về ARXIV_DELAY (= ARXIV_DELAY: tắt)
KEEP_ARCHIVE = False            # True: giữ thêm file nén cạnh .tex/.bib (tốn đĩa)
METADATA_BATCH_SIZE = 100       # số bài mỗi request metadata arXiv (tối đa 100)
S2_BATCH_SIZE = 100             # số bài mỗi request batch Semantic Scholar
S2_DELAY_WITH_KEY = 1.0         # có S2_API_KEY: giây giữa hai request S2 (key bảo đảm 1 request/giây);
                                # không có key: giữ 3 giây mặc định của get_extracting_reference
RETRY_NOT_FOUND = False         # True: thử lại các bài / references 'not_found' của phiên trước
MAX_CONSECUTIVE_ERRORS = 5      # số lỗi liên tiếp trước khi dừng một làn
COOLDOWNS_SEC = [300, 600, 900] # arXiv vẫn trả 429/5xx sau khi module tự chờ: lần nghỉ thứ k dài COOLDOWNS_SEC[k] giây
                                # rồi thử lại cùng request; hết danh sách thì dừng làn arXiv
SOURCE_WRITE_QUEUE = 3          # số version đã tải chờ giải nén/ghi ở thread nền (0 = ghi ngay trong làn arXiv)
MAX_RUN_HOURS = 11.5            # tự dừng gọn trước khi Colab ngắt phiên (~12h); None = không giới hạn
INTERRUPT_GRACE = 60.0          # giây chờ các làn xong việc hiện tại khi bấm Interrupt

MONITOR_EVERY = 1.0             # giây giữa hai mẫu RSS
DISK_EVERY = 300.0              # giây giữa hai lần đo dung lượng data/ (quét cả thư mục, chậm trên Drive)
PROGRESS_EVERY = 600.0          # giây giữa hai dòng tiến độ in ra notebook
SOURCE_STATE_FLUSH_EVERY = 120.0 # giây; gom các lần ghi source_state.json (0 = ghi sau mỗi version)

PERSIST_TO_DRIVE = True         # Colab: data/ nằm trên Google Drive để chạy tiếp qua nhiều phiên
DRIVE_DATA_DIR = "/content/drive/MyDrive/Intro2DS-Group3/data"  # thư mục chung của nhóm (shortcut) hoặc Drive riêng
MIN_FREE_DISK_GB = 5.0          # cảnh báo khi chỗ trống dưới ngưỡng

In [ ]:
# 5.2 — Kiểm tra range + cấu hình, nạp S2_API_KEY (phải trước khi import get_extracting_reference)
if RUN_RANGES == "all":
    RANGES_TO_RUN = list(RANGES)
else:
    RANGES_TO_RUN = sorted(set([RUN_RANGES] if isinstance(RUN_RANGES, int) else RUN_RANGES))
if not RANGES_TO_RUN or any(r not in RANGES for r in RANGES_TO_RUN):
    raise ValueError(f"RUN_RANGES phải gồm các số trong {list(RANGES)} hoặc là 'all' (đang là {RUN_RANGES!r}).")
# Tên thư mục state/log của máy này, tự sinh từ range: vd R3, R1+R2, all
WORKER = "all" if RANGES_TO_RUN == sorted(RANGES) else "+".join(f"R{r}" for r in RANGES_TO_RUN)
if ARXIV_DELAY < 3.0:
    raise ValueError("Đề yêu cầu chờ ít nhất 3 giây giữa hai request arXiv.")
if ARXIV_MAX_DELAY < ARXIV_DELAY:
    raise ValueError("ARXIV_MAX_DELAY phải >= ARXIV_DELAY.")
if SAMPLE_N is not None and SAMPLE_N < 1:
    raise ValueError("SAMPLE_N phải >= 1 hoặc None.")
if not COOLDOWNS_SEC or min(COOLDOWNS_SEC) <= 0:
    raise ValueError("COOLDOWNS_SEC phải là danh sách số giây > 0.")
if not 1 <= METADATA_BATCH_SIZE <= 100:
    raise ValueError("METADATA_BATCH_SIZE phải trong khoảng 1..100.")
if S2_DELAY_WITH_KEY < 1.0:
    raise ValueError("Semantic Scholar có key chỉ bảo đảm 1 request/giây: S2_DELAY_WITH_KEY phải >= 1.")

VERSIONS_PER_PAPER = 31530 / 25000  # discovery 2026-10-26: 31.530 version / 25.000 bài (range 5 cao hơn)
total = 0
print(f"Máy này ({WORKER}) chạy:")
for label in RANGES_TO_RUN:
    yymm, start, end = RANGES[label]
    n = end - start + 1 if SAMPLE_N is None else min(SAMPLE_N, end - start + 1)
    total += n
    print(f"  range {label}  {yymm}.{start:05d} -> {yymm}.{end:05d}   {n} bài")
print(f"Tổng: {total} bài | Phase B ước tính ~{total * VERSIONS_PER_PAPER * ARXIV_DELAY / 3600:.1f} giờ "
      f"(~{VERSIONS_PER_PAPER:.2f} version/bài × {ARXIV_DELAY:g} s, chưa tính 429)")


def load_s2_key():
    """Đặt os.environ['S2_API_KEY'] từ biến môi trường hoặc Colab Secrets; trả về nguồn (không in key)."""
    if os.environ.get("S2_API_KEY"):
        return "biến môi trường"
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("S2_API_KEY")
        except Exception:
            key = None
        if key:
            os.environ["S2_API_KEY"] = key
            return "Colab Secrets"
    return None


S2_KEY_SOURCE = load_s2_key()
print("S2_API_KEY:", S2_KEY_SOURCE or "không có (dùng pool chung, dễ bị 429)")

## 6. Lưu trữ dữ liệu và module

`utils.DATA_DIR` cố định là `<repo>/data`. Trên Colab, ô 6.1 biến `<repo>/data` thành **symlink sang Google Drive** (không sửa `utils.py`), để dữ liệu và state còn lại sau khi phiên Colab kết thúc. State và log của phiên ghi vào thư mục con `WORKER`; `gfs.STATE_FILE` được trỏ vào đó.

Ô 6.2 import các module của nhóm. Nếu có `S2_API_KEY` thì ô này đặt nhịp Semantic Scholar về `S2_DELAY_WITH_KEY`. Log chi tiết của các module được ghi vào file; notebook chỉ hiện cảnh báo.

In [ ]:
# 6.1 — data/ (Colab: symlink sang Drive), thư mục state/log của WORKER
DATA_DIR = REPO_ROOT / "data"

if IN_COLAB and PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DATA_DIR)
    target.mkdir(parents=True, exist_ok=True)
    if DATA_DIR.is_symlink():
        if DATA_DIR.resolve() != target.resolve():
            raise RuntimeError(f"{DATA_DIR} đang trỏ tới {DATA_DIR.resolve()}, không phải {target}.")
    else:
        if DATA_DIR.exists():
            if any(DATA_DIR.iterdir()):
                raise RuntimeError(f"{DATA_DIR} đã có dữ liệu local; hãy chuyển sang {target} trước.")
            DATA_DIR.rmdir()
        DATA_DIR.symlink_to(target, target_is_directory=True)
DATA_DIR.mkdir(exist_ok=True)

STATE_ROOT = DATA_DIR / ".state"            # mỗi máy (WORKER) một thư mục con
LOG_ROOT = DATA_DIR / "run_logs"
STATE_DIR = STATE_ROOT / WORKER
LOG_DIR = LOG_ROOT / WORKER
TABLE_DIR = DATA_DIR / "tables"
EXPORT_DIR = DATA_DIR / "report_exports"
for d in (STATE_DIR, LOG_DIR, TABLE_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SOURCE_STATE_FILE = STATE_DIR / "source_state.json"  # state của get_full_source
REFS_DONE_FILE = STATE_DIR / "refs_done.jsonl"       # kết quả references từng bài (định dạng của get_extracting_reference)
NB_STATUS_FILE = STATE_DIR / "nb_status.jsonl"       # kết quả metadata/source do notebook ghi (dòng sau ghi đè dòng trước)
RUNS_FILE = LOG_DIR / "runs.jsonl"          # một dòng start + một dòng end cho mỗi phiên
TIMINGS_FILE = LOG_DIR / "timings.jsonl"    # thời gian từng (bài, stage)
EVENTS_FILE = LOG_DIR / "events.jsonl"      # sự kiện bất thường (dừng làn, batch lỗi, cooldown, ...)
SCRAPER_LOG = LOG_DIR / "scraper.log"       # log chi tiết của các module
# Không thuộc corpus khi tính dung lượng output (source_state.json thẳng trong data/ là của bản cũ)
NON_CORPUS = {".state", "run_logs", "tables", "report_exports", "source_state.json", "source_state.tmp"}

free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"data/ = {DATA_DIR} -> {DATA_DIR.resolve()}")
print(f"State / log của máy này ({WORKER}): {STATE_DIR.relative_to(DATA_DIR)}, {LOG_DIR.relative_to(DATA_DIR)}")
print(f"Còn trống: {free_gb:.1f} GB")
if free_gb < MIN_FREE_DISK_GB:
    print(f"CẢNH BÁO: dưới {MIN_FREE_DISK_GB} GB.")

In [ ]:
# 6.2 — Import module của nhóm, cấu hình logging một lần
import logging

import arxiv
import requests

import utils
import get_metadata as gm
import get_full_source as gfs
import get_extracting_reference as ger

if not hasattr(gm, "get_entry_discovery_batch") or not hasattr(gfs, "Pacer"):
    raise RuntimeError(
        f"{SRC_DIR} đang là bản module cũ (thiếu get_metadata.get_entry_discovery_batch / get_full_source.Pacer). "
        "Trên Colab: push bản mới nhất của src/ lên GitHub, xoá thư mục repo đã clone "
        f"(!rm -rf {REPO_ROOT}) rồi Restart session và chạy lại."
    )
if Path(utils.DATA_DIR).resolve() != DATA_DIR.resolve():
    raise RuntimeError(f"utils.DATA_DIR ({utils.DATA_DIR}) khác DATA_DIR của notebook ({DATA_DIR}).")
gfs.STATE_FILE = SOURCE_STATE_FILE  # module đọc biến này lúc chạy
# Module đọc key lúc import; gán lại phòng khi module đã được import trước khi có key
if os.environ.get("S2_API_KEY"):
    ger._SESSION.headers["x-api-key"] = os.environ["S2_API_KEY"]
# Nhịp Semantic Scholar: có key -> S2_DELAY_WITH_KEY; không có key -> mặc định của module (module đọc biến này lúc gọi)
if not hasattr(ger, "_nb_default_interval"):
    ger._nb_default_interval = ger._MIN_INTERVAL
ger._MIN_INTERVAL = S2_DELAY_WITH_KEY if os.environ.get("S2_API_KEY") else ger._nb_default_interval
print(f"Semantic Scholar: {ger._MIN_INTERVAL:g} s giữa hai request ({'có' if os.environ.get('S2_API_KEY') else 'không có'} key)")


def setup_logging(log_file: Path) -> None:
    """Các module đều gọi basicConfig lúc import: thay bằng log INFO ra file, notebook chỉ hiện WARNING."""
    root = logging.getLogger()
    for h in list(root.handlers):
        root.removeHandler(h)
        h.close()
    fmt = logging.Formatter("%(asctime)s | %(threadName)-12s | %(name)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(fmt)
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.WARNING)
    console.setFormatter(fmt)
    root.addHandler(file_handler)
    root.addHandler(console)
    root.setLevel(logging.INFO)
    for noisy in ("arxiv", "urllib3"):
        logging.getLogger(noisy).setLevel(logging.WARNING)


setup_logging(SCRAPER_LOG)
log = logging.getLogger("scraper_nb")
print("Log chi tiết:", SCRAPER_LOG)

## 7. Các thành phần hỗ trợ

Mục này **chỉ định nghĩa hàm và lớp**: không gọi mạng, không ghi dữ liệu.

### 7.1 Ghi log và đọc trạng thái

Tiến độ không lưu trong biến của notebook mà luôn được **dựng lại từ các file state trên đĩa**. Các file đó là: `nb_status.jsonl` (kết quả metadata / source), `refs_done.jsonl` (kết quả references), `source_state.json` (trạng thái từng version) và `metadata.json` của từng bài. `read_status` đọc gộp state của mọi máy và của bản cũ, nên dữ liệu đã gộp vẫn được tính đúng. `build_todo` lập danh sách việc còn lại cho từng làn.

In [ ]:
# 7.1 — Ghi log & đọc trạng thái
import csv
import threading
from collections import Counter, defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd

_io_lock = threading.Lock()


def append_jsonl(path: Path, record: dict) -> None:
    """Ghi nối một dòng JSON và flush ngay (runtime bị ngắt cũng không mất dòng đã ghi)."""
    line = json.dumps(record, ensure_ascii=False)
    with _io_lock, open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()


def read_jsonl(path: Path) -> list:
    """Đọc jsonl, bỏ qua dòng hỏng (dòng ghi dở khi runtime bị ngắt)."""
    if not path.exists():
        return []
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out


def generate_ids(ranges: dict, labels: list, sample_n=None) -> pd.DataFrame:
    """DataFrame (arxiv_id, range_label); số thứ tự đệm 5 chữ số, vd 2608.04007."""
    rows = []
    for label in labels:
        yymm, start, end = ranges[label]
        stop = end if sample_n is None else min(end, start + sample_n - 1)
        rows += [(f"{yymm}.{n:05d}", label) for n in range(start, stop + 1)]
    df = pd.DataFrame(rows, columns=["arxiv_id", "range_label"])
    if df["arxiv_id"].duplicated().any():
        raise ValueError("Các range bị trùng ID.")
    return df


def read_metadata_file(arxiv_id: str):
    try:
        return json.loads((utils.get_paper_dir(arxiv_id) / "metadata.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def state_paths(name: str, legacy: Path = None) -> list:
    """File state `name` của mọi máy, theo thứ tự: bản cũ (trước khi chia theo range), của máy
    khác, của WORKER. Đọc lần lượt và để bản ghi sau đè bản ghi trước, nên bản ghi của WORKER được ưu tiên."""
    others = sorted(p for p in STATE_ROOT.glob(f"*/{name}") if p.parent != STATE_DIR)
    return [legacy or STATE_ROOT / name, *others, STATE_DIR / name]


def read_refs_done() -> dict:
    """Bản ghi cuối của từng bài trong các refs_done.jsonl."""
    last = {}
    for path in state_paths("refs_done.jsonl"):
        for r in read_jsonl(path):
            if isinstance(r.get("id"), str):
                last[r["id"]] = r
    return last


def read_nb_status() -> dict:
    """{(stage, arxiv_id): bản ghi cuối} từ các nb_status.jsonl."""
    return {(r["stage"], r["arxiv_id"]): r for path in state_paths("nb_status.jsonl") for r in read_jsonl(path)}


def load_source_papers() -> dict:
    """`papers` của source_state gộp từ mọi máy; phần của WORKER lấy từ RAM (mới nhất)."""
    papers = {}
    for path in state_paths("source_state.json", legacy=DATA_DIR / "source_state.json")[:-1]:
        try:
            papers.update(json.loads(path.read_text(encoding="utf-8")).get("papers", {}))
        except (OSError, json.JSONDecodeError):
            pass
    with gfs.STATE_LOCK:  # làn arXiv có thể đang thêm bài vào state
        papers.update(gfs.load_state().get("papers", {}))
    return papers


def source_status(entry) -> str:
    """Trạng thái source của một bài từ source_state."""
    if not entry:
        return "pending"
    if entry.get("completed"):
        # completed = mọi version có trạng thái kết thúc; "tex_bib" = đã extract xong
        return "done" if "tex_bib" in entry.get("sizes", {}) else "downloaded"
    return "partial" if entry.get("versions") else "pending"


def read_status(ids_df: pd.DataFrame) -> pd.DataFrame:
    """Trạng thái mỗi bài × mỗi phase, dựng lại hoàn toàn từ các file state trên đĩa."""
    nb = read_nb_status()
    refs = read_refs_done()
    src_papers = load_source_papers()
    on_disk = {e.name for e in os.scandir(DATA_DIR) if e.is_dir()}
    rows = []
    for aid in ids_df["arxiv_id"]:
        m = nb.get(("metadata", aid), {})
        meta_status, num_versions = m.get("status", "pending"), m.get("num_versions")
        if meta_status != "done" and utils.get_paper_dir(aid).name in on_disk:
            meta = read_metadata_file(aid)  # metadata.json có thể do chạy module trực tiếp mà có
            if meta and meta.get("arxiv_id") and meta.get("title"):
                meta_status, num_versions = "done", meta.get("num_versions")
        src = source_status(src_papers.get(aid))
        if src != "done" and nb.get(("source", aid), {}).get("status") == "failed":
            src = "failed"
        r = refs.get(aid)
        rows.append({
            "arxiv_id": aid,
            "metadata": meta_status,
            "num_versions": num_versions,
            "source": src,
            "refs": r["status"] if r else "pending",
            "n_refs": r.get("n_refs") if r else None,
        })
    return ids_df.merge(pd.DataFrame(rows), on="arxiv_id", how="left")


def build_todo(status_df: pd.DataFrame) -> dict:
    """Việc của phiên này: danh sách cho làn arXiv và làn S2."""
    retry = {"pending", "failed"} | ({"not_found"} if RETRY_NOT_FOUND else set())
    df = status_df.copy()
    df["need_metadata"] = df["metadata"].isin(retry) & RUN_METADATA
    df["need_source"] = (df["source"] != "done") & (df["need_metadata"] | (df["metadata"] == "done")) & RUN_SOURCE
    arxiv_todo = df.loc[df["need_metadata"] | df["need_source"],
                        ["arxiv_id", "need_metadata", "need_source", "num_versions"]]
    refs_todo = df.loc[df["refs"].isin(retry), "arxiv_id"].tolist() if RUN_REFS else []
    return {"arxiv": arxiv_todo.to_dict("records"), "refs": refs_todo}


def summarize_status(status_df: pd.DataFrame) -> pd.DataFrame:
    """Bảng đếm trạng thái theo range × phase."""
    parts = []
    for stage in ("metadata", "source", "refs"):
        t = pd.crosstab(status_df["range_label"], status_df[stage])
        t.columns = pd.MultiIndex.from_product([[stage], t.columns])
        parts.append(t)
    table = pd.concat(parts, axis=1).fillna(0).astype(int)
    table.loc["TOTAL"] = table.sum()
    return table


CONFIG_KEYS = ["RUN_RANGES", "WORKER", "RANGES_TO_RUN", "SAMPLE_N", "RUN_METADATA", "RUN_SOURCE", "RUN_REFS",
               "PARALLEL", "REPORT_ALL_RANGES", "ARXIV_DELAY", "ARXIV_MAX_DELAY", "KEEP_ARCHIVE", "S2_BATCH_SIZE", "S2_DELAY_WITH_KEY",
               "RETRY_NOT_FOUND", "MAX_CONSECUTIVE_ERRORS", "COOLDOWNS_SEC", "SOURCE_WRITE_QUEUE", "METADATA_BATCH_SIZE",
               "MAX_RUN_HOURS", "MONITOR_EVERY", "DISK_EVERY", "SOURCE_STATE_FLUSH_EVERY", "PERSIST_TO_DRIVE"]


def run_config() -> dict:
    return {k: globals()[k] for k in CONFIG_KEYS}


def fmt_bytes(n) -> str:
    if n is None or (isinstance(n, float) and math.isnan(n)):
        return "-"
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024:
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def fmt_secs(s: float) -> str:
    h, rem = divmod(int(s), 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}"

### 7.2 Đo thời gian và tài nguyên

- `RunContext.timed(stage, arxiv_id)`: ghi thời gian của từng cặp (bài, stage) vào `timings.jsonl`, kể cả khi lỗi.
- `ResourceMonitor`: thread nền ghi **RSS của tiến trình** (`psutil`, gồm cả tiến trình con) vào `monitor_<run_id>.csv`, và **dung lượng `data/`** (quét như `du -sb`) vào `disk_<run_id>.csv`. Hai việc chạy ở hai thread riêng để lần quét thư mục lớn không làm trễ mẫu RSS.
- `peak_rss_bytes()`: RSS đỉnh do hệ điều hành ghi nhận (`ru_maxrss` trên Linux), để đối chiếu với số liệu lấy mẫu.

In [ ]:
# 7.2 — Đo đạc
def dir_size(path: Path, exclude=frozenset()) -> int:
    """Tổng st_size mọi file dưới `path` (như `du -sb`); bỏ các file / thư mục cấp 1 có tên trong `exclude`."""
    root, total, stack = str(path), 0, [str(path)]
    while stack:
        cur = stack.pop()
        try:
            with os.scandir(cur) as it:
                for e in it:
                    if cur == root and e.name in exclude:
                        continue
                    if e.is_dir(follow_symlinks=False):
                        stack.append(e.path)
                    elif e.is_file(follow_symlinks=False):
                        total += e.stat(follow_symlinks=False).st_size
        except OSError:
            pass  # file/thư mục bị xoá trong lúc quét (archive tạm)
    return total


def peak_rss_bytes():
    """RSS đỉnh của kernel từ lúc khởi động (Linux: ru_maxrss; Windows: peak working set)."""
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    except ImportError:
        return getattr(psutil.Process().memory_info(), "peak_wset", None)


class RunContext:
    """Trạng thái dùng chung của một phiên: run_id, cờ dừng, bộ đếm, ghi timings/status/events."""

    def __init__(self, run_id: str, max_hours=None):
        self.run_id = run_id
        self.stop_event = threading.Event()
        self.deadline = time.monotonic() + max_hours * 3600 if max_hours else None
        self.time_limit_hit = False
        self.counters = defaultdict(Counter)  # stage -> Counter(status)
        self.lane_stops = {}                  # lane -> lý do dừng sớm
        self.phase = {}                       # lane -> việc đang làm
        self.totals = {}                      # lane -> số việc của phiên
        self.progress = {}                    # lane -> số việc đã xử lý

    def should_stop(self) -> bool:
        if not self.stop_event.is_set() and self.deadline and time.monotonic() > self.deadline:
            self.time_limit_hit = True
            self.stop_event.set()
            self.event("all", "time_limit", f"Đạt MAX_RUN_HOURS={MAX_RUN_HOURS}")
        return self.stop_event.is_set()

    def phase_text(self) -> str:
        return " | ".join(f"{k}: {v}" for k, v in sorted(self.phase.items()))

    @contextmanager
    def timed(self, stage: str, arxiv_id: str, **extra):
        """Ghi {stage, arxiv_id, start, end, duration, status} vào timings.jsonl, kể cả khi có exception."""
        rec = {"run_id": self.run_id, "stage": stage, "arxiv_id": arxiv_id, "start": time.time(), **extra}
        t0, status = time.perf_counter(), "ok"
        try:
            yield
        except BaseException as e:
            status = getattr(e, "status", None) or f"error:{type(e).__name__}"
            raise
        finally:
            rec.update(end=time.time(), duration=round(time.perf_counter() - t0, 3), status=status)
            append_jsonl(TIMINGS_FILE, rec)

    def log_timing(self, rec: dict) -> None:
        append_jsonl(TIMINGS_FILE, {"run_id": self.run_id, **rec})

    def mark(self, stage: str, arxiv_id: str, status: str, **extra) -> None:
        """Ghi kết quả metadata/source vào nb_status.jsonl (dùng để resume)."""
        self.counters[stage][status] += 1
        append_jsonl(NB_STATUS_FILE, {"run_id": self.run_id, "stage": stage, "arxiv_id": arxiv_id,
                                      "status": status, "t": time.time(), **extra})

    def count(self, stage: str, status: str) -> None:
        self.counters[stage][status] += 1

    def event(self, lane: str, kind: str, message: str, **extra) -> None:
        log.warning("[%s] %s: %s", lane, kind, message)
        append_jsonl(EVENTS_FILE, {"run_id": self.run_id, "t": time.time(), "lane": lane,
                                   "kind": kind, "message": message, **extra})

    def stop_lane(self, lane: str, reason: str) -> None:
        self.lane_stops[lane] = reason
        self.event(lane, "lane_stop", reason)


class ResourceMonitor:
    """Lấy mẫu RSS mỗi `every` giây và dung lượng data/ mỗi `disk_every` giây, ghi CSV dạng append."""

    def __init__(self, run_id: str, every: float, disk_every: float, phase_fn=lambda: ""):
        self.proc = psutil.Process()
        self.every, self.disk_every, self.phase_fn = every, disk_every, phase_fn
        self.rss_path = LOG_DIR / f"monitor_{run_id}.csv"
        self.disk_path = LOG_DIR / f"disk_{run_id}.csv"
        self._stop = threading.Event()
        self._threads = []
        self.rss_last = self.rss_max = self.rss_sum = self.rss_n = 0
        self.disk_last = self.disk_max = None

    def rss_now(self) -> int:
        rss = self.proc.memory_info().rss
        for child in self.proc.children(recursive=True):
            try:
                rss += child.memory_info().rss
            except psutil.Error:
                pass
        return rss

    def _sample_rss(self) -> list:
        rss = self.rss_now()
        self.rss_last, self.rss_max = rss, max(self.rss_max, rss)
        self.rss_sum += rss
        self.rss_n += 1
        return [round(time.time(), 3), rss, self.phase_fn()]

    def _sample_disk(self) -> list:
        t0 = time.perf_counter()
        size = dir_size(DATA_DIR)
        self.disk_last, self.disk_max = size, max(self.disk_max or 0, size)
        return [round(time.time(), 3), size, round(time.perf_counter() - t0, 2)]

    def _loop(self, path: Path, header: list, interval: float, sample) -> None:
        new = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(header)
            stopping = False
            while not stopping:
                w.writerow(sample())
                f.flush()
                stopping = self._stop.wait(interval)
            w.writerow(sample())  # mẫu cuối lúc dừng
            f.flush()

    def start(self) -> None:
        self._stop.clear()
        self._threads = [
            threading.Thread(target=self._loop, name="monitor-rss", daemon=True,
                             args=(self.rss_path, ["t", "rss_bytes", "phase"], self.every, self._sample_rss)),
            threading.Thread(target=self._loop, name="monitor-disk", daemon=True,
                             args=(self.disk_path, ["t", "data_dir_bytes", "scan_sec"], self.disk_every, self._sample_disk)),
        ]
        for t in self._threads:
            t.start()

    def stop(self) -> None:
        self._stop.set()
        for t in self._threads:
            t.join()

    def summary(self) -> dict:
        return {
            "rss_max_bytes": self.rss_max,
            "rss_mean_bytes": round(self.rss_sum / self.rss_n) if self.rss_n else None,
            "rss_samples": self.rss_n,
            "disk_max_bytes": self.disk_max,
            "disk_last_bytes": self.disk_last,
        }

### 7.3 Lớp gọi module

Đây là lớp duy nhất gọi các module của nhóm. Mọi exception của module được quy về các loại ở mục 3.4.

- `SourceStateCache`: `get_full_source` checkpoint state sau mỗi version. Notebook thay `gfs.load_state` / `gfs.save_state` (chỉ trong kernel, không sửa file module) bằng bản giữ state trong RAM. Bản này ghi xuống đĩa tối đa mỗi `SOURCE_STATE_FLUSH_EVERY` giây, cộng một lần khi kết thúc phiên. Đặt `SOURCE_STATE_FLUSH_EVERY = 0` để ghi sau mỗi version.
- `fetch_metadata_batch`, `fetch_metadata`: Phase A theo batch, hoặc từng bài khi batch lỗi.
- `fetch_source`: Phase B qua `gfs.scrape_paper`. Nếu dùng thread ghi, hàm trả về khi đã **tải** xong; kết quả **ghi** (`done` / `failed`) được báo về sau qua callback.
- `fetch_refs_batch`: Phase C. Kết quả ghi vào `refs_done.jsonl` của `WORKER`, cùng định dạng với `get_extracting_reference.mark_done`.

In [ ]:
# 7.3 — Lớp gọi module (adapter) của nhóm
class PaperError(Exception):
    """Lỗi riêng của một bài: ghi lại rồi đi tiếp. status: 'not_found' | 'failed'."""

    def __init__(self, status: str, message: str = ""):
        super().__init__(message or status)
        self.status = status


class ServiceStop(Exception):
    """Dịch vụ từ chối. retryable=False (403 / hết quota): dừng cả làn, phiên sau chạy tiếp.
    retryable=True (429 / 5xx kéo dài): làn arXiv nghỉ theo COOLDOWNS_SEC rồi thử lại cùng request."""
    status = "service_stop"

    def __init__(self, message: str = "", retryable: bool = False):
        super().__init__(message)
        self.retryable = retryable


# Giữ bản gốc một lần duy nhất, để chạy lại ô này không bọc chồng lên bản đã bọc
if not hasattr(gfs, "_nb_original_state_io"):
    gfs._nb_original_state_io = (gfs.load_state, gfs.save_state)
_ORIG_LOAD_STATE, _ORIG_SAVE_STATE = gfs._nb_original_state_io


class SourceStateCache:
    """State của get_full_source giữ trong RAM, ghi xuống đĩa tối đa mỗi `flush_every` giây."""

    def __init__(self, flush_every: float):
        self.flush_every = flush_every
        self.state = None
        self.dirty = False
        self.last_flush = time.monotonic()
        self.lock = gfs.STATE_LOCK  # chung với module: không ghi file giữa lúc state đang đổi

    def load(self) -> dict:
        with self.lock:
            if self.state is None:
                self.state = _ORIG_LOAD_STATE()
            return self.state

    def save(self, state: dict) -> None:
        with self.lock:
            self.state, self.dirty = state, True
            if time.monotonic() - self.last_flush >= self.flush_every:
                self.flush()

    def flush(self) -> None:
        with self.lock:
            if self.dirty and self.state is not None:
                _ORIG_SAVE_STATE(self.state)
                self.dirty = False
            self.last_flush = time.monotonic()


if SOURCE_STATE_FLUSH_EVERY > 0:
    SOURCE_STATE = SourceStateCache(SOURCE_STATE_FLUSH_EVERY)
    gfs.load_state, gfs.save_state = SOURCE_STATE.load, SOURCE_STATE.save
else:
    SOURCE_STATE = None
    gfs.load_state, gfs.save_state = _ORIG_LOAD_STATE, _ORIG_SAVE_STATE

SRC_SESSION = requests.Session()  # giữ kết nối keep-alive tới export.arxiv.org


def arxiv_api_error(e: arxiv.HTTPError) -> Exception:
    """Lỗi HTTP của arXiv API (module đã tự retry; tới đây là lỗi thật) -> ServiceStop / PaperError."""
    status = getattr(e, "status", None)
    if status == 403:
        return ServiceStop(f"arXiv API HTTP {status}")
    if status == 429 or (status or 0) >= 500:
        return ServiceStop(f"arXiv API HTTP {status}", retryable=True)
    return PaperError("failed", f"arXiv API HTTP {status}")


def fetch_metadata(arxiv_id: str) -> dict:
    """Phase A (từng bài): get_entry_discovery + save_metadata."""
    try:
        meta = gm.get_entry_discovery(arxiv_id)
    except ValueError as e:  # module báo không tìm thấy bài
        raise PaperError("not_found", str(e)) from e
    except arxiv.HTTPError as e:
        raise arxiv_api_error(e) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    gm.save_metadata(arxiv_id, meta)
    return meta


def fetch_metadata_batch(ids: list) -> dict:
    """Phase A (batch): get_entry_discovery_batch + save_metadata. Trả {arxiv_id: metadata | None (not_found)}."""
    try:
        metas = gm.get_entry_discovery_batch(ids)
    except arxiv.HTTPError as e:
        raise arxiv_api_error(e) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    for aid, meta in metas.items():
        if meta:
            gm.save_metadata(aid, meta)
    return metas


def fetch_source(arxiv_id: str, num_versions: int, pacer: gfs.Pacer, writer, on_finished) -> dict:
    """Phase B: tải qua /src/ mọi version còn thiếu, chỉ giữ .tex/.bib (giải nén từ RAM).
    Có writer: trả về khi đã TẢI xong, kết quả ghi (done / failed) báo qua on_finished(entry, error) ở thread nền.
    Không có writer: ghi xong rồi mới trả về, on_finished được gọi trước khi trả về."""
    try:
        entry = gfs.scrape_paper(arxiv_id, num_versions, pacer=pacer, session=SRC_SESSION, keep_archive=KEEP_ARCHIVE,
                                 writer=writer, on_finished=on_finished)
    except gfs.BlockedError as e:  # 403
        raise ServiceStop(str(e)) from e
    except gfs.RateLimitError as e:  # vẫn 429 / 5xx sau khi module tự chờ
        raise ServiceStop(str(e), retryable=True) from e
    except Exception as e:  # DownloadError (mạng, archive hỏng) và lỗi khác: thử lại ở phiên sau
        raise PaperError("failed", repr(e)) from e
    return entry


def mark_refs(arxiv_id: str, status: str, n_refs: int) -> None:
    """Như ger.mark_done, nhưng ghi vào refs_done.jsonl của WORKER."""
    append_jsonl(REFS_DONE_FILE, {"id": arxiv_id, "status": status, "n_refs": n_refs})


def fetch_refs_batch(ids: list) -> dict:
    """Phase C: một request batch; lưu references.json + refs_done.jsonl. Trả {arxiv_id: status}."""
    try:
        results = ger._fetch_batch(ids)
    except ger.QuotaExhausted as e:
        raise ServiceStop(str(e)) from e
    except Exception as e:
        raise PaperError("failed", repr(e)) from e
    out = {}
    for pid in ids:
        refs = results.get(pid)
        if refs is None:
            out[pid] = "not_found"
        elif not refs:
            out[pid] = "no_refs"
        else:
            ger.save_extracting_references(pid, refs)
            out[pid] = "ok"
        mark_refs(pid, out[pid], len(refs or []))
    return out

## 8. Thu thập dữ liệu

### 8.1 Danh sách ID và tình trạng hiện tại

Bảng dưới được dựng lại từ các file state; nó cho biết lần chạy này sẽ tiếp tục từ đâu.

| Phase | Trạng thái |
|---|---|
| `metadata` | `pending`, `done`, `not_found` (arXiv không có bài này), `failed` (sẽ thử lại) |
| `source` | `pending`, `partial` (tải dở), `downloaded` (chưa extract), `done`, `failed` |
| `refs` | `pending`, `ok`, `no_refs`, `not_found` (Semantic Scholar chưa có bài) |

In [ ]:
# 8.1 — ID của các range được chọn + tình trạng từ state
ids_df = generate_ids(RANGES, RANGES_TO_RUN, SAMPLE_N)
status_df = read_status(ids_df)
todo_preview = build_todo(status_df)

print(f"{len(ids_df)} ID | còn việc: làn arXiv {len(todo_preview['arxiv'])} bài, làn S2 {len(todo_preview['refs'])} bài")
display(summarize_status(status_df))

### 8.2 Làn arXiv: entry discovery và full-text source

Một thread, tuần tự, một kết nối, một nhịp `gfs.Pacer` chung cho mọi request:

1. **Phase A — entry discovery theo batch** cho mọi bài còn thiếu metadata. Mỗi request `get_entry_discovery_batch` lấy bản mới nhất của tới `METADATA_BATCH_SIZE` bài; hậu tố `vN` của bản mới nhất cho biết số version. Bài nhiều version cần thêm request gom `v1..vN` để có ngày của từng version. Kết quả là `metadata.json` của từng bài. Bài không có trong kết quả là `not_found`. Nếu cả batch lỗi thì từng bài tự lấy metadata riêng bằng `get_entry_discovery`.
2. **Phase B — full-text source** cho từng bài: tải `v1..vN` qua `/src/` của `export.arxiv.org`, chỉ giữ `.tex` / `.bib`. Việc giải nén và ghi do thread `source-writer` làm, nên làn chuyển ngay sang request kế tiếp; cuối làn chờ thread này ghi nốt. Version chỉ có PDF (hoặc docx, …) giữ một thư mục rỗng.

Nếu arXiv vẫn trả 429 / 5xx sau khi module tự chờ, làn **nghỉ theo `COOLDOWNS_SEC`** rồi thử lại cùng request (Interrupt hoặc `MAX_RUN_HOURS` cắt ngang được lúc nghỉ). Gặp 403 thì làn dừng ngay. Mỗi lần nghỉ được ghi thành một sự kiện `cooldown` trong `events.jsonl`.

**Tổng thời gian entry discovery** bằng tổng thời lượng các stage `metadata` trong `timings.jsonl`. Thời gian của một batch được chia đều cho các bài trong batch (ghi kèm `amortized: true`).

In [ ]:
# 8.2 — Làn arXiv
class LaneInterrupted(Exception):
    """Interrupt / MAX_RUN_HOURS trong lúc đang nghỉ cooldown."""


def sleep_unless_stopped(ctx: RunContext, seconds: float) -> bool:
    """Nghỉ `seconds` giây; dậy sớm khi Interrupt hoặc đạt MAX_RUN_HOURS. True nếu nghỉ đủ."""
    end = time.monotonic() + seconds
    while not ctx.should_stop():
        left = end - time.monotonic()
        if left <= 0:
            return True
        ctx.stop_event.wait(min(left, 30.0))  # thức dậy định kỳ để should_stop() kiểm tra deadline
    return False


def with_cooldown(ctx: RunContext, label: str, fn):
    """Gọi fn(). arXiv 429/5xx kéo dài (ServiceStop retryable) -> nghỉ COOLDOWNS_SEC[0], [1], ... giây rồi gọi lại;
    hết danh sách hoặc 403 -> ServiceStop có kèm `label`. Bị dừng lúc nghỉ -> LaneInterrupted."""
    cooldowns = 0
    while True:
        try:
            return fn()
        except ServiceStop as e:
            if not e.retryable or cooldowns >= len(COOLDOWNS_SEC):
                raise ServiceStop(f"{label}: {e}" + (f" (sau {cooldowns} lần cooldown)" if cooldowns else "")) from e
            pause = COOLDOWNS_SEC[cooldowns]
            cooldowns += 1
            ctx.count("cooldown", "arxiv")
            ctx.event("arxiv", "cooldown",
                      f"{label}: {e}; nghỉ {pause:.0f}s rồi thử lại ({cooldowns}/{len(COOLDOWNS_SEC)})", seconds=pause)
            ctx.phase["arxiv"] = f"cooldown {label} ({cooldowns}/{len(COOLDOWNS_SEC)})"
            if not sleep_unless_stopped(ctx, pause):
                raise LaneInterrupted() from e


def timed(ctx: RunContext, stage: str, aid: str, fn, *args, **extra):
    with ctx.timed(stage, aid, **extra):
        return fn(*args)


def num_versions_of(item: dict):
    """num_versions của một việc (từ Phase A, từ trạng thái, hoặc đọc lại metadata.json); None nếu không có."""
    nv = item.get("num_versions")
    if nv is None or pd.isna(nv):
        nv = (read_metadata_file(item["arxiv_id"]) or {}).get("num_versions")
    return int(nv) if nv else None


def prefetch_metadata(ctx: RunContext, pacer: gfs.Pacer, chunk: list) -> None:
    """Phase A theo batch cho các bài trong `chunk` (sửa item tại chỗ).
    Batch lỗi (PaperError): giữ nguyên need_metadata để từng bài tự lấy metadata riêng."""
    ids = [it["arxiv_id"] for it in chunk]
    label = f"batch {ids[0]}..{ids[-1]}"
    ctx.phase["arxiv"] = f"A {label} ({len(ids)} bài)"

    def attempt():
        t0 = time.perf_counter()
        return pacer.call(fetch_metadata_batch, ids), time.perf_counter() - t0

    t_start = time.time()
    try:
        metas, duration = with_cooldown(ctx, label, attempt)
    except PaperError as e:
        ctx.event("arxiv", "metadata_batch_failed", f"{label}: {e}; chuyển sang lấy từng bài")
        return
    t_end = time.time()
    for it in chunk:
        aid, meta = it["arxiv_id"], metas.get(it["arxiv_id"])
        ctx.log_timing({"stage": "metadata", "arxiv_id": aid, "start": t_start, "end": t_end,
                        "duration": round(duration / len(chunk), 3), "status": "ok" if meta else "not_found",
                        "batch_size": len(chunk), "amortized": True})
        it["need_metadata"] = False
        if meta:
            it["num_versions"] = meta["num_versions"]
            ctx.mark("metadata", aid, "done", num_versions=meta["num_versions"])
        else:  # arXiv bỏ qua id không tồn tại
            it["need_source"] = False
            ctx.mark("metadata", aid, "not_found", error="không có trong kết quả batch")


def arxiv_lane(ctx: RunContext, todo: list) -> None:
    pacer = gfs.Pacer(ARXIV_DELAY, ARXIV_MAX_DELAY)  # một nhịp chung cho API metadata và /src/
    writer = gfs.Writer(SOURCE_WRITE_QUEUE) if SOURCE_WRITE_QUEUE > 0 else None  # giải nén/ghi ở thread nền
    ctx.pacer, ctx.writer = pacer, writer  # để dòng tiến độ và dòng kết thúc phiên đọc
    consecutive_errors = 0

    def attempt(stage: str, aid: str, fn) -> bool:
        """Một việc của một bài; lỗi riêng của bài được ghi lại. False nếu làn phải dừng."""
        nonlocal consecutive_errors
        try:
            fn()
            consecutive_errors = 0
        except PaperError as e:
            ctx.mark(stage, aid, e.status, error=str(e)[:500])
            if e.status != "failed":  # not_found là kết quả hợp lệ
                consecutive_errors = 0
                return True
            consecutive_errors += 1
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                ctx.stop_lane("arxiv", f"{consecutive_errors} lỗi liên tiếp, lỗi cuối ({aid}): {e}")
                return False
        return True

    def metadata_one(it: dict) -> None:
        aid = it["arxiv_id"]
        ctx.phase["arxiv"] = f"A {aid}"
        meta = with_cooldown(ctx, aid, lambda: timed(ctx, "metadata", aid, pacer.call, fetch_metadata, aid))
        it["need_metadata"], it["num_versions"] = False, meta["num_versions"]
        ctx.mark("metadata", aid, "done", num_versions=meta["num_versions"])

    def stored(aid: str, nv: int):
        """Callback khi mọi version của bài đã ghi xong (chạy ở thread ghi nếu có writer)."""
        def on_finished(entry: dict, error) -> None:
            if error is not None:  # vd archive hỏng: thử lại ở phiên sau
                ctx.mark("source", aid, "failed", error=repr(error)[:500])
            elif entry.get("completed"):
                ctx.mark("source", aid, "done", num_versions=nv)
            else:
                ctx.mark("source", aid, "failed", error="chưa tải đủ mọi version")
        return on_finished

    def source_one(it: dict) -> None:
        aid, nv = it["arxiv_id"], it["num_versions"]
        ctx.phase["arxiv"] = f"B {aid} ({nv} version)"
        with_cooldown(ctx, aid, lambda: timed(ctx, "source", aid, fetch_source, aid, nv, pacer, writer,
                                              stored(aid, nv), num_versions=nv))

    try:
        # Phase A: metadata của mọi bài trước, để biết số version
        need = [it for it in todo if it["need_metadata"]]
        ctx.totals["arxiv A"], ctx.progress["arxiv A"] = len(need), 0
        for start in range(0, len(need), METADATA_BATCH_SIZE):
            if ctx.should_stop():
                return
            prefetch_metadata(ctx, pacer, need[start:start + METADATA_BATCH_SIZE])
            ctx.progress["arxiv A"] = min(start + METADATA_BATCH_SIZE, len(need))
        for it in need:
            if it["need_metadata"]:  # batch của nó bị lỗi
                if ctx.should_stop():
                    return
                if not attempt("metadata", it["arxiv_id"], lambda: metadata_one(it)):
                    return

        # Phase B: mọi version của từng bài
        work = []
        for it in todo:
            if it["need_source"] and not it["need_metadata"]:
                it["num_versions"] = num_versions_of(it)
                if it["num_versions"]:
                    work.append(it)
                else:
                    ctx.mark("source", it["arxiv_id"], "failed", error="không đọc được num_versions")
        ctx.totals["arxiv B"], ctx.progress["arxiv B"] = len(work), 0
        for it in work:
            if ctx.should_stop():
                return
            if not attempt("source", it["arxiv_id"], lambda: source_one(it)):
                return
            ctx.progress["arxiv B"] += 1
    except ServiceStop as e:  # 403, hoặc vẫn 429/5xx sau mọi lần nghỉ trong COOLDOWNS_SEC
        ctx.stop_lane("arxiv", str(e))
    except LaneInterrupted:  # Interrupt / hết giờ lúc đang nghỉ: không coi là dịch vụ từ chối
        pass
    finally:
        if writer is not None:
            ctx.phase["arxiv"] = f"chờ ghi nốt {writer.pending} version đã tải"
            writer.close()
        ctx.phase["arxiv"] = "dừng"

### 8.3 Làn Semantic Scholar: references

Mỗi request batch lấy reference của `S2_BATCH_SIZE` bài (endpoint `/paper/batch`). Bài có từ 1.000 reference trở lên được lấy đủ qua endpoint phân trang. Mọi reference được lưu trong `references.json` của bài; kết quả từng bài ghi vào `refs_done.jsonl`. Thời gian mỗi bài bằng thời gian của batch chia cho số bài trong batch.

In [ ]:
# 8.3 — Làn Semantic Scholar
def s2_lane(ctx: RunContext, ids: list) -> None:
    consecutive_errors = 0
    n_batches = math.ceil(len(ids) / S2_BATCH_SIZE)
    ctx.totals["s2"] = len(ids)
    for b, start in enumerate(range(0, len(ids), S2_BATCH_SIZE), 1):
        if ctx.should_stop():
            break
        batch = ids[start:start + S2_BATCH_SIZE]
        ctx.phase["s2"] = f"C batch {b}/{n_batches}"
        t_start, t0 = time.time(), time.perf_counter()
        stop_reason = None
        try:
            results = fetch_refs_batch(batch)
            consecutive_errors = 0
        except ServiceStop as e:
            results, stop_reason = {}, str(e)
        except PaperError as e:
            results = {pid: "failed" for pid in batch}
            consecutive_errors += 1
            ctx.event("s2", "batch_failed", f"batch {b} ({batch[0]}...): {e}")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                stop_reason = f"{consecutive_errors} batch lỗi liên tiếp"
        duration, t_end = time.perf_counter() - t0, time.time()
        for pid, status in results.items():
            ctx.count("refs", status)
            ctx.log_timing({"stage": "refs", "arxiv_id": pid, "start": t_start, "end": t_end,
                            "duration": round(duration / len(batch), 3), "status": status,
                            "batch_size": len(batch), "amortized": True})
        ctx.progress["s2"] = start + len(batch)
        if stop_reason:
            ctx.stop_lane("s2", stop_reason)
            break
    ctx.phase["s2"] = "dừng"

### 8.4 Điều phối một phiên

`run(ids_df)` là **một phiên** cào:

1. Đọc state, lập danh sách việc còn lại.
2. Bật monitor, chạy làn arXiv ‖ làn Semantic Scholar (`PARALLEL = False` để chạy tuần tự khi cần so sánh).
3. In tiến độ mỗi `PROGRESS_EVERY` giây: số việc đã xong, nhịp arXiv hiện tại, số lần retry, số version chờ ghi, RAM, dung lượng `data/`.
4. Kết thúc khi hết việc, khi dịch vụ từ chối, khi đạt `MAX_RUN_HOURS` hoặc khi bấm *Interrupt*. Dù dừng vì lý do nào, notebook vẫn ghi state và ghi dòng kết thúc phiên vào `runs.jsonl`.

In [ ]:
# 8.4 — Điều phối một phiên
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

# Các làn của phiên trước (có thể còn chạy nền sau Interrupt); giữ qua các lần chạy lại ô này
ACTIVE_LANES = globals().get("ACTIVE_LANES", [])

def print_progress(ctx: RunContext, monitor: ResourceMonitor, t0: float) -> None:
    lanes = " | ".join(f"{lane} {ctx.progress.get(lane, 0)}/{n}" for lane, n in ctx.totals.items())
    pacer, writer = getattr(ctx, "pacer", None), getattr(ctx, "writer", None)
    pace = f" | nhịp arXiv {pacer.interval:.1f}s" if pacer else ""
    if pacer and pacer.events:
        pace += " (" + ", ".join(f"{k}={v}" for k, v in sorted(pacer.events.items())) + ")"
    if writer:
        pace += f" | chờ ghi {writer.pending}"
    print(f"[{time.strftime('%H:%M:%S')}] {fmt_secs(time.time() - t0)} | {lanes}{pace} | "
          f"RSS {fmt_bytes(monitor.rss_last)} | data/ {fmt_bytes(monitor.disk_last)}")
    for stage, c in ctx.counters.items():
        print(f"    {stage}: " + ", ".join(f"{k}={v}" for k, v in sorted(c.items())))


def run(ids_df: pd.DataFrame) -> dict:
    """Một phiên cào. Chạy lại để tiếp tục từ state."""
    leftover = [f for f in ACTIVE_LANES if not f.done()]
    if leftover:  # không bao giờ để hai làn arXiv chạy cùng lúc
        print(f"Đợi {len(leftover)} làn của phiên trước dừng hẳn...")
        wait(leftover)
    todo = build_todo(read_status(ids_df))
    if not todo["arxiv"] and not todo["refs"]:
        print("Không còn việc: xem kết quả ở mục 9–11.")
        return {"run_id": None, "stop_reason": "nothing_to_do"}

    run_id = f"{WORKER}-{time.strftime('%Y%m%d-%H%M%S')}"
    ctx = RunContext(run_id, MAX_RUN_HOURS)
    monitor = ResourceMonitor(run_id, MONITOR_EVERY, DISK_EVERY, ctx.phase_text)
    print(f"run_id={run_id} | làn arXiv: {len(todo['arxiv'])} bài | làn S2: {len(todo['refs'])} bài")
    append_jsonl(RUNS_FILE, {"run_id": run_id, "event": "start", "t": time.time(), "worker": WORKER,
                             "ranges": RANGES_TO_RUN, "machine": machine_info(),
                             "config": run_config(), "todo": {k: len(v) for k, v in todo.items()}})
    monitor.start()
    t0, stop_reason = time.time(), "completed"
    executor = ThreadPoolExecutor(max_workers=2 if PARALLEL else 1, thread_name_prefix="lane")
    futures, wait_lanes = [], True
    try:
        if todo["arxiv"]:
            futures.append(executor.submit(arxiv_lane, ctx, todo["arxiv"]))
        if todo["refs"]:
            futures.append(executor.submit(s2_lane, ctx, todo["refs"]))
        ACTIVE_LANES[:] = futures
        pending = set(futures)
        while pending:
            done, pending = wait(pending, timeout=PROGRESS_EVERY, return_when=FIRST_COMPLETED)
            for f in done:
                f.result()  # lỗi lập trình trong một làn -> dừng cả phiên
            print_progress(ctx, monitor, t0)
    except KeyboardInterrupt:
        stop_reason = "interrupted"
        ctx.stop_event.set()
        print(f"Đang dừng: chờ tối đa {INTERRUPT_GRACE:.0f}s để các làn xong việc hiện tại "
              "(bấm Interrupt lần nữa để thôi chờ)...")
        try:
            _, still_running = wait(futures, timeout=INTERRUPT_GRACE)
        except KeyboardInterrupt:
            still_running = [f for f in futures if not f.done()]
        if still_running:
            # Module đang tự retry bên trong (có thể vài phút): không chờ nữa; làn sẽ tự dừng sau việc hiện tại
            wait_lanes = False
            stop_reason = "interrupted (làn còn chạy nền tới hết việc hiện tại)"
    except BaseException as e:
        stop_reason = f"error: {e!r}"
        ctx.stop_event.set()
        raise
    finally:
        executor.shutdown(wait=wait_lanes)
        if SOURCE_STATE is not None:
            SOURCE_STATE.flush()
        monitor.stop()
        if stop_reason == "completed" and ctx.time_limit_hit:
            stop_reason = "time_limit"
        elif stop_reason == "completed" and ctx.lane_stops:
            stop_reason = "lane_stop"
        end_rec = {"run_id": run_id, "event": "end", "t": time.time(), "wall_sec": round(time.time() - t0, 2),
                   "stop_reason": stop_reason, "lane_stops": ctx.lane_stops,
                   "counters": {k: dict(v) for k, v in ctx.counters.items()},
                   "monitor": monitor.summary(), "peak_rss_kernel_bytes": peak_rss_bytes(),
                   "arxiv_delay_last": getattr(getattr(ctx, "pacer", None), "interval", None),
                   "arxiv_events": dict(getattr(getattr(ctx, "pacer", None), "events", {})),
                   "writer_busy_sec": round(getattr(getattr(ctx, "writer", None), "busy_sec", 0.0), 2)}
        append_jsonl(RUNS_FILE, end_rec)
        print(f"Kết thúc phiên {run_id}: {stop_reason} sau {fmt_secs(end_rec['wall_sec'])}")
        if ctx.lane_stops:
            print("  Làn dừng sớm:", ctx.lane_stops)
        print("  RSS max:", fmt_bytes(monitor.rss_max), "| data/ max:", fmt_bytes(monitor.disk_max))
    return end_rec

### 8.5 Chạy

Ô dưới chạy một phiên. Bị dừng thì **chạy lại ô này** (hoặc *Run all* ở phiên Colab mới) để tiếp tục. Ô kế tiếp in tình trạng sau phiên.

In [ ]:
# 8.5 — CHẠY (chạy lại ô này để tiếp tục sau khi bị dừng)
last_run = run(ids_df)

In [ ]:
# 8.5 — Tình trạng sau phiên
status_df = read_status(ids_df)
display(summarize_status(status_df))

## 9. Kết quả: dữ liệu thu được

Mục 9–11 **chỉ đọc đĩa**. Chạy sau khi cào xong để chốt số liệu, hoặc chạy giữa chừng để xem tiến độ. `REPORT_ALL_RANGES = False` thì chỉ tính các range và log của máy này. `REPORT_ALL_RANGES = True` thì tính cả 5 range và log của mọi máy trong `data/run_logs/`; dùng khi `data/` đã có dữ liệu cả nhóm.

### 9.1 Bảng `papers` và `references`

- `papers`: 6 cột theo đề. Các cột thêm (`num_versions`, `categories`, `abstract`) nằm trong `papers_full` và trong `per_paper.csv`.
- `references_all`: **mọi** reference crawl được (đề yêu cầu giữ ở local).
- `references`: chỉ những reference có `ref_arxiv_id`, đã bỏ cặp `(arxiv_id, ref_arxiv_id)` trùng. Đây là bảng sẽ nộp lên Hugging Face.

In [ ]:
# 9.1 — Dựng bảng từ metadata.json / references.json
from concurrent.futures import ThreadPoolExecutor

from tqdm import tqdm  # thanh tiến độ dạng text: chạy được cả khi không có ipywidgets

PAPERS_COLUMNS = ["arxiv_id", "title", "authors", "submission_date", "revised_dates", "venue"]
PAPERS_EXTRA = ["num_versions", "categories", "abstract"]
REF_COLUMNS = ["arxiv_id", "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date", "ref_s2_id"]


def read_json_many(paths: list, workers: int = 16) -> list:
    """Đọc nhiều file JSON song song (Drive chậm khi đọc tuần tự từng file nhỏ)."""
    def load(p):
        try:
            return json.loads(Path(p).read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return None
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(tqdm(pool.map(load, paths), total=len(paths), leave=False))


REPORT_RANGES = list(RANGES) if REPORT_ALL_RANGES else RANGES_TO_RUN
report_ids_df = generate_ids(RANGES, REPORT_RANGES, SAMPLE_N)
# Log hiệu năng: của máy này, hoặc của mọi máy khi báo cáo cả nhóm
REPORT_LOG_DIRS = sorted(p for p in LOG_ROOT.iterdir() if p.is_dir()) if REPORT_ALL_RANGES else [LOG_DIR]
print(f"Phạm vi: {len(report_ids_df)} ID của {REPORT_RANGES} | log của {[p.name for p in REPORT_LOG_DIRS]}")

status_df = read_status(report_ids_df)

meta_ids = status_df.loc[status_df["metadata"] == "done", "arxiv_id"].tolist()
metas = [m for m in read_json_many([utils.get_paper_dir(a) / "metadata.json" for a in meta_ids]) if m]
papers_full = pd.DataFrame(metas, columns=PAPERS_COLUMNS + PAPERS_EXTRA).merge(report_ids_df, on="arxiv_id", how="left")
papers = papers_full[PAPERS_COLUMNS].reset_index(drop=True)

ok_ids = status_df.loc[status_df["refs"] == "ok", "arxiv_id"].tolist()
ref_lists = read_json_many([utils.get_paper_dir(a) / "references.json" for a in ok_ids])
references_all = pd.DataFrame([r for refs in ref_lists if refs for r in refs], columns=REF_COLUMNS)
with_arxiv = references_all[references_all["ref_arxiv_id"].notna()]
references = with_arxiv.drop_duplicates(["arxiv_id", "ref_arxiv_id"]).reset_index(drop=True)
n_ref_duplicates = len(with_arxiv) - len(references)

print(f"papers: {len(papers)} | references_all: {len(references_all)} | references (có arXiv ID): {len(references)}")
display(papers.head(3))
display(references.head(3))

### 9.2 Kiểm tra schema

Kiểm tra mỗi bảng: khoá không rỗng và không trùng, ngày ở dạng ISO, tác giả là danh sách, và tổng số reference khớp với state.

In [ ]:
# 9.2 — Kiểm tra schema
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}")


def is_iso(d) -> bool:
    return isinstance(d, str) and bool(ISO_DATE.match(d))


def validate_tables() -> pd.DataFrame:
    expected_refs = pd.to_numeric(status_df.loc[status_df["refs"] == "ok", "n_refs"], errors="coerce").sum()
    checks = [
        ("papers", "arxiv_id null", papers["arxiv_id"].isna().sum()),
        ("papers", "arxiv_id trùng", papers["arxiv_id"].duplicated().sum()),
        ("papers", "title rỗng", (papers["title"].fillna("").str.len() == 0).sum()),
        ("papers", "authors không phải list", (~papers["authors"].map(lambda a: isinstance(a, list))).sum()),
        ("papers", "submission_date không phải ISO", (~papers["submission_date"].map(is_iso)).sum()),
        ("papers", "revised_dates có ngày không phải ISO",
         papers["revised_dates"].map(lambda ds: not all(is_iso(d) for d in (ds or []))).sum()),
        ("references", "arxiv_id / ref_arxiv_id null", references[["arxiv_id", "ref_arxiv_id"]].isna().any(axis=1).sum()),
        ("references", "cặp (arxiv_id, ref_arxiv_id) trùng (đã bỏ)", n_ref_duplicates),
        ("references_all", "lệch giữa số dòng và tổng n_refs trong refs_done", abs(len(references_all) - int(expected_refs))),
    ]
    table = pd.DataFrame(checks, columns=["bảng", "kiểm tra", "số vi phạm"])
    table["số vi phạm"] = table["số vi phạm"].astype(int)
    return table


display(validate_tables())
old_style = references["ref_arxiv_id"].astype(str).str.contains("/").sum()
print(f"ref_arxiv_id kiểu cũ (archive/YYMMNNN, vd hep-th/9901001), giữ nguyên: {old_style}")

### 9.3 Lưu bảng

Ghi ba bảng ra `data/tables/*.parquet`, rồi đọc lại để kiểm tra số dòng và tên cột.

In [ ]:
# 9.3 — Ghi parquet và đọc lại để kiểm tra
TABLE_FILES = {
    "papers": (papers, TABLE_DIR / "papers.parquet"),
    "references": (references, TABLE_DIR / "references.parquet"),
    "references_all": (references_all, TABLE_DIR / "references_all.parquet"),
}
for name, (df, path) in TABLE_FILES.items():
    df.to_parquet(path, index=False)
    back = pd.read_parquet(path)
    assert len(back) == len(df) and list(back.columns) == list(df.columns), name
    print(f"{path.name:24s} {len(back):>9} dòng  {fmt_bytes(path.stat().st_size):>10}")

### 9.4 Kiểm tra corpus: chỉ còn `.tex` / `.bib`

Đề yêu cầu bỏ hình bằng cách **extract chọn lọc**: mỗi thư mục version chỉ được chứa `.tex` / `.bib`. Thư mục rỗng nghĩa là version đó chỉ có PDF hoặc định dạng khác.

In [ ]:
# 9.4 — Quét thư mục version
ARCHIVE_NAMES = set(gfs.ARCHIVE_NAME.values())


def scan_paper_corpus(arxiv_id: str):
    out = {"arxiv_id": arxiv_id, "n_version_dirs": 0, "n_empty_versions": 0,
           "n_tex": 0, "n_bib": 0, "tex_bib_bytes": 0, "n_other": 0}
    bad = []
    pdir = utils.get_paper_dir(arxiv_id)
    for vdir in pdir.glob("v[0-9]*") if pdir.is_dir() else []:
        out["n_version_dirs"] += 1
        files = [f for f in vdir.rglob("*") if f.is_file()]
        if not files:
            out["n_empty_versions"] += 1
        for f in files:
            name = f.name.lower()
            if name.endswith((".tex", ".bib")):
                out["n_tex" if name.endswith(".tex") else "n_bib"] += 1
                out["tex_bib_bytes"] += f.stat().st_size
            elif not (KEEP_ARCHIVE and f.parent == vdir and f.name in ARCHIVE_NAMES):
                out["n_other"] += 1
                bad.append(f.relative_to(DATA_DIR).as_posix())
    return out, bad


scan_ids = status_df.loc[status_df["source"] != "pending", "arxiv_id"].tolist()
with ThreadPoolExecutor(max_workers=16) as pool:
    scanned = list(tqdm(pool.map(scan_paper_corpus, scan_ids), total=len(scan_ids), leave=False))
corpus_df = pd.DataFrame([s for s, _ in scanned], columns=list(scan_paper_corpus("x")[0]))
violations = [p for _, bad in scanned for p in bad]

print(f"Bài đã quét: {len(corpus_df)} | thư mục version: {corpus_df['n_version_dirs'].sum()} "
      f"(rỗng: {corpus_df['n_empty_versions'].sum()}) | .tex: {corpus_df['n_tex'].sum()} | .bib: {corpus_df['n_bib'].sum()}")
print(f"File không phải .tex/.bib: {len(violations)} (mong đợi 0)")
for p in violations[:20]:
    print("  ", p)

### 9.5 Thống kê dữ liệu

| Chỉ số | Định nghĩa |
|---|---|
| Bài cào thành công | có metadata **và** source `done` (mọi version đã được tải và extract) |
| Tỷ lệ thành công | trên số ID trong range, và trên số bài arXiv thực sự có (metadata khác `not_found`) |
| Kích thước trước / sau khi bỏ hình | tổng byte mọi file trong archive (`sizes.full`) / tổng byte `.tex` + `.bib` giữ lại (`sizes.tex_bib`); tính trên các bài có source TeX |
| Số reference trung bình | trên các bài Semantic Scholar trả reference (`ok`); kèm con số tính trên mọi bài đã hỏi (`no_refs` / `not_found` tính là 0) |
| Tỷ lệ lấy reference thành công | số bài `ok` / số bài đã hỏi Semantic Scholar |

In [ ]:
# 9.5 — Bảng per_paper (một dòng mỗi ID) + thống kê dữ liệu
def jsonable(obj):
    """Đổi kiểu numpy/pandas và NaN sang kiểu JSON chuẩn."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if obj is pd.NA or obj is pd.NaT:
        return None
    if isinstance(obj, np.generic):
        obj = obj.item()
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj


def mean_or_none(s: pd.Series):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.mean()) if len(s) else None


def ratio(a, b):
    return float(a) / float(b) if b else None


def source_info(entry) -> dict:
    entry = entry or {}
    st = Counter(v.get("status") for v in entry.get("versions", {}).values())
    sizes, stats = entry.get("sizes", {}), entry.get("stats", {})

    def total(stage, key):
        return sizes.get(stage, {}).get("totals", {}).get(key)

    return {"n_versions_done": st.get("done", 0), "n_versions_no_tex": st.get("no_tex", 0),
            "n_versions_not_found": st.get("not_found", 0),
            "archive_bytes": total("archive_only", "archive_bytes"),
            "size_full_bytes": total("full", "extracted_bytes"),
            "size_tex_bib_bytes": total("tex_bib", "extracted_bytes"),
            "source_sleep_sec": stats.get("sleep_sec")}


timings = pd.DataFrame([r for d in REPORT_LOG_DIRS for r in read_jsonl(d / "timings.jsonl")])
if timings.empty:
    timings = pd.DataFrame(columns=["run_id", "stage", "arxiv_id", "start", "end", "duration", "status"])
stage_time = (timings.groupby(["arxiv_id", "stage"])["duration"].sum().unstack()
              .reindex(columns=["metadata", "source", "refs"])
              .rename(columns=lambda s: f"time_{s}_sec"))

src_state = load_source_papers()
per_paper = status_df.copy()
per_paper = per_paper.join(pd.DataFrame([source_info(src_state.get(a)) for a in per_paper["arxiv_id"]],
                                        index=per_paper.index))
n_refs_arxiv = references_all.groupby("arxiv_id")["ref_arxiv_id"].count().rename("n_refs_arxiv")
per_paper = (per_paper.merge(n_refs_arxiv, left_on="arxiv_id", right_index=True, how="left")
             .merge(stage_time, left_on="arxiv_id", right_index=True, how="left"))
per_paper["time_total_sec"] = per_paper[[c for c in per_paper if c.startswith("time_")]].sum(axis=1, min_count=1)
per_paper["num_versions"] = pd.to_numeric(per_paper["num_versions"], errors="coerce")
per_paper["n_refs"] = pd.to_numeric(per_paper["n_refs"], errors="coerce")
per_paper["paper_success"] = (per_paper["metadata"] == "done") & (per_paper["source"] == "done")


def data_stats(pp: pd.DataFrame) -> dict:
    found = pp["metadata"] == "done"
    has_tex = pp["size_full_bytes"].notna() & (pd.to_numeric(pp["size_full_bytes"], errors="coerce") > 0)
    asked = pp["refs"].isin(["ok", "no_refs", "not_found"])
    before = pd.to_numeric(pp.loc[has_tex, "size_full_bytes"]).sum()
    after = pd.to_numeric(pp.loc[has_tex, "size_tex_bib_bytes"], errors="coerce").sum()
    n_refs_sum = pp.loc[pp["refs"] == "ok", "n_refs"].sum()
    return {
        "n_ids": len(pp),
        "n_papers_found": int(found.sum()),
        "n_metadata_not_found": int((pp["metadata"] == "not_found").sum()),
        "n_papers_success": int(pp["paper_success"].sum()),
        "success_rate_over_ids": ratio(pp["paper_success"].sum(), len(pp)),
        "success_rate_over_found": ratio(pp["paper_success"].sum(), found.sum()),
        "n_versions_total": int(pp.loc[found, "num_versions"].sum()),
        "n_versions_done": int(pp["n_versions_done"].sum()),
        "n_versions_no_tex": int(pp["n_versions_no_tex"].sum()),
        "n_versions_not_found": int(pp["n_versions_not_found"].sum()),
        "n_papers_with_tex": int(has_tex.sum()),
        "avg_archive_bytes": mean_or_none(pp.loc[has_tex, "archive_bytes"]),
        "avg_size_before_bytes": mean_or_none(pp.loc[has_tex, "size_full_bytes"]),
        "avg_size_after_bytes": mean_or_none(pp.loc[has_tex, "size_tex_bib_bytes"]),
        "size_reduction_ratio": 1 - after / before if before else None,
        "n_refs_asked": int(asked.sum()),
        "refs_status_counts": pp["refs"].value_counts().to_dict(),
        "refs_success_rate": ratio((pp["refs"] == "ok").sum(), asked.sum()),
        "avg_refs_per_paper_ok": mean_or_none(pp.loc[pp["refs"] == "ok", "n_refs"]),
        "avg_refs_per_paper_asked": mean_or_none(pp.loc[asked, "n_refs"].fillna(0)),
        "n_references_all": int(n_refs_sum) if n_refs_sum == n_refs_sum else 0,
        "avg_refs_with_arxiv_per_paper_ok": mean_or_none(pp.loc[pp["refs"] == "ok", "n_refs_arxiv"].fillna(0)),
        "ref_arxiv_coverage": ratio(pp["n_refs_arxiv"].sum(), n_refs_sum),
    }


DATA_STATS = data_stats(per_paper)
range_stats = pd.DataFrame({label: data_stats(g) for label, g in per_paper.groupby("range_label")}).T
range_stats = range_stats.drop(columns=["refs_status_counts"])
versions_dist = per_paper.loc[per_paper["metadata"] == "done", "num_versions"].value_counts().sort_index()
primary_cat = papers_full["categories"].map(lambda c: c[0] if isinstance(c, list) and c else None)
category_counts = primary_cat.value_counts().rename_axis("primary_category").rename("n_papers")

pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))
display(pd.Series(jsonable(DATA_STATS), name="giá trị").to_frame())
display(range_stats)
print("Phân bố số version:", versions_dist.to_dict())
display(category_counts.head(15).to_frame())

## 10. Kết quả: hiệu năng

### 10.1 Số liệu

| Chỉ số | Cách tính |
|---|---|
| Thời gian xử lý một bài | tổng thời lượng các stage của bài trong `timings.jsonl` (metadata và refs chia đều theo batch), cộng dồn qua các phiên |
| Thời gian trung bình mỗi bài | trung bình / median / p95 của chỉ số trên, tính trên các bài thành công; kèm thời gian máy / số ID |
| Tổng thời gian entry discovery | tổng thời lượng mọi stage `metadata` |
| Wall time end-to-end | mỗi máy (thư mục `run_logs/<WORKER>/`): tổng (end − start) các phiên trong `runs.jsonl`. Phiên bị ngắt đột ngột lấy mẫu monitor cuối cùng làm end. `wall_time_sec` = giá trị lớn nhất giữa các máy (các máy chạy song song); `machine_time_sec` = tổng của mọi máy |
| Retry của arXiv | số lần 429, 5xx, tải bị đứt, lỗi mạng (từ `runs.jsonl`), cùng số lần và tổng thời gian cooldown (từ `events.jsonl`) |
| RAM tối đa / trung bình | max / mean `rss_bytes` qua mọi `monitor_*.csv` (mỗi mẫu là RSS của **một** instance), kèm `ru_maxrss` của kernel |
| Disk tối đa | max `data_dir_bytes` qua mọi `disk_*.csv` |
| Dung lượng output cuối | dung lượng `data/` trừ `run_logs/`, `.state/`, `tables/`, `report_exports/` |

In [ ]:
# 10.1 — Hiệu năng (cộng dồn mọi phiên của các máy trong phạm vi)
def load_csv_logs(prefix: str) -> pd.DataFrame:
    frames = []
    for d in REPORT_LOG_DIRS:
        for p in sorted(d.glob(f"{prefix}_*.csv")):
            try:
                df = pd.read_csv(p, on_bad_lines="skip")
            except pd.errors.EmptyDataError:
                continue
            df["worker"], df["run_id"] = d.name, p.stem.split("_", 1)[1]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["t", "worker", "run_id"])


def load_runs(monitor: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for d in REPORT_LOG_DIRS:
        recs = read_jsonl(d / "runs.jsonl")
        ends = {r["run_id"]: r for r in recs if r.get("event") == "end"}
        for s in (r for r in recs if r.get("event") == "start"):
            rid, e = s["run_id"], ends.get(s["run_id"])
            if e:
                end_t, reason = e["t"], e.get("stop_reason")
            else:  # runtime bị ngắt trước khi ghi dòng end
                last = monitor.loc[monitor["run_id"] == rid, "t"].max() if len(monitor) else None
                end_t = last if last == last and last is not None else s["t"]
                reason = "killed (end = mẫu monitor cuối)"
            m = s.get("machine", {})
            rows.append({"worker": d.name, "run_id": rid, "start": s["t"], "end": end_t, "wall_sec": end_t - s["t"],
                         "stop_reason": reason, "todo_arxiv": s.get("todo", {}).get("arxiv"),
                         "todo_refs": s.get("todo", {}).get("refs"), "colab": m.get("colab"),
                         "cpu_model": m.get("cpu_model"), "cpu_count": m.get("cpu_count"),
                         "ram_total_bytes": m.get("ram_total_bytes"),
                         "peak_rss_kernel_bytes": (e or {}).get("peak_rss_kernel_bytes")})
    return pd.DataFrame(rows, columns=["worker", "run_id", "start", "end", "wall_sec", "stop_reason", "todo_arxiv",
                                       "todo_refs", "colab", "cpu_model", "cpu_count", "ram_total_bytes",
                                       "peak_rss_kernel_bytes"])


monitor_df = load_csv_logs("monitor")
disk_df = load_csv_logs("disk")
runs_df = load_runs(monitor_df)

ok_time = per_paper.loc[per_paper["paper_success"], "time_total_sec"].dropna()
worker_wall = runs_df.groupby("worker")["wall_sec"].sum()
wall = float(worker_wall.max()) if len(worker_wall) else 0.0          # các máy chạy song song
machine_time = float(worker_wall.sum()) if len(worker_wall) else 0.0
output_bytes = dir_size(DATA_DIR, exclude=NON_CORPUS)

# Retry của arXiv: bộ đếm của Pacer (dòng end trong runs.jsonl) + các lần cooldown (events.jsonl)
arxiv_events = Counter()
for d in REPORT_LOG_DIRS:
    for r in read_jsonl(d / "runs.jsonl"):
        if r.get("event") == "end":
            arxiv_events.update(r.get("arxiv_events") or {})
cooldowns = [r for d in REPORT_LOG_DIRS for r in read_jsonl(d / "events.jsonl") if r.get("kind") == "cooldown"]

PERF_STATS = {
    "n_workers": len(worker_wall),
    "n_sessions": len(runs_df),
    "wall_time_sec": wall,
    "machine_time_sec": machine_time,
    "wall_time_per_worker_sec": worker_wall.to_dict(),
    "time_per_paper_mean_sec": float(ok_time.mean()) if len(ok_time) else None,
    "time_per_paper_median_sec": float(ok_time.median()) if len(ok_time) else None,
    "time_per_paper_p95_sec": float(ok_time.quantile(0.95)) if len(ok_time) else None,
    "machine_time_per_id_sec": machine_time / len(per_paper) if len(per_paper) else None,
    "entry_discovery_total_sec": float(timings.loc[timings["stage"] == "metadata", "duration"].sum()),
    "source_total_sec": float(timings.loc[timings["stage"] == "source", "duration"].sum()),
    "refs_total_sec": float(timings.loc[timings["stage"] == "refs", "duration"].sum()),
    "arxiv_retry_events": dict(arxiv_events),
    "n_cooldowns": len(cooldowns),
    "cooldown_sec_total": float(sum(r.get("seconds", 0) for r in cooldowns)),
    "throughput_papers_per_hour": per_paper["paper_success"].sum() / (wall / 3600) if wall else None,
    "rss_max_bytes": float(monitor_df["rss_bytes"].max()) if "rss_bytes" in monitor_df else None,
    "rss_mean_bytes": float(monitor_df["rss_bytes"].mean()) if "rss_bytes" in monitor_df else None,
    "peak_rss_kernel_bytes": float(runs_df["peak_rss_kernel_bytes"].max()) if len(runs_df) else None,
    "disk_max_bytes": float(disk_df["data_dir_bytes"].max()) if "data_dir_bytes" in disk_df else None,
    "output_size_bytes": output_bytes,
    "tables_size_bytes": dir_size(TABLE_DIR),
}

display(pd.Series(jsonable(PERF_STATS), name="giá trị").to_frame())
print(f"Wall time: {fmt_secs(wall)} ({len(worker_wall)} máy song song, {len(runs_df)} phiên) | "
      f"thời gian máy {fmt_secs(machine_time)} | RSS max {fmt_bytes(PERF_STATS['rss_max_bytes'])} | "
      f"disk max {fmt_bytes(PERF_STATS['disk_max_bytes'])} | output {fmt_bytes(output_bytes)}")
display(runs_df)

### 10.2 Biểu đồ kiểm tra nhanh

Ba biểu đồ: RSS theo thời gian, phân bố thời gian xử lý mỗi bài, và kích thước trung bình mỗi bài trước / sau khi bỏ hình. Đây chỉ là bản xem nhanh; hình chính thức do Jupyter Book vẽ từ `data/report_exports/` (mục 11).

In [ ]:
# 10.2 — Vài hình kiểm tra nhanh (hình chính thức do Jupyter Book vẽ từ report_exports/)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for worker, m in monitor_df.sort_values("t").groupby("worker"):
    axes[0].plot((m["t"] - m["t"].min()) / 3600, m["rss_bytes"] / 2**20, lw=0.8, label=worker)
if monitor_df["worker"].nunique() > 1:
    axes[0].legend()
axes[0].set(title="RSS theo thời gian", xlabel="giờ (từ phiên đầu của mỗi người)", ylabel="MB")
if len(ok_time):
    axes[1].hist(ok_time, bins=40)
axes[1].set(title="Thời gian xử lý mỗi bài thành công", xlabel="giây", ylabel="số bài")
sizes = [DATA_STATS["avg_size_before_bytes"] or 0, DATA_STATS["avg_size_after_bytes"] or 0]
axes[2].bar(["trước khi bỏ hình", "chỉ .tex/.bib"], [s / 2**10 for s in sizes])
axes[2].set(title="Kích thước trung bình mỗi bài", ylabel="KB")
plt.tight_layout()
plt.show()

## 11. Xuất số liệu cho Jupyter Book

Các file nhỏ trong `data/report_exports/` được chép sang repo Jupyter Book. Book đọc các file này và **vẽ hình lúc build**, vì đề không chấp nhận ảnh chụp dán vào. Ý nghĩa từng file được ghi trong `report_exports/README.md`.

In [ ]:
# 11 — Ghi report_exports/
EXPORT_README = """# report_exports

Số liệu do `src/scraper.ipynb` (mục 9–11) sinh ra, dùng cho Jupyter Book.

| File | Nội dung |
|---|---|
| `summary_stats.json` | `data` (thống kê dữ liệu, mục 9.5), `performance` (hiệu năng, mục 10.1), `versions_distribution`, `config`, `generated_at` |
| `per_paper.csv` | một dòng mỗi arXiv ID: `range_label` (số range 1–5); trạng thái `metadata` / `source` / `refs`; `num_versions`, `n_versions_done/no_tex/not_found`; `archive_bytes` (nén), `size_full_bytes` (trước khi bỏ hình), `size_tex_bib_bytes` (sau); `n_refs`, `n_refs_arxiv`; `time_metadata_sec`, `time_source_sec`, `time_refs_sec` (chia đều theo batch), `time_total_sec`; `source_sleep_sec`; `paper_success` |
| `range_stats.csv` | các chỉ số của `summary_stats.data` tính riêng cho từng range |
| `ranges.csv` | `range_label`, `yymm`, `start`, `end` |
| `category_counts.csv` | số bài theo category chính (category đầu tiên) |
| `runs.csv` | mỗi phiên chạy: `worker` (thư mục log của máy), `run_id`, `start`, `end` (epoch giây), `wall_sec`, `stop_reason`, việc của phiên, thông tin máy, `peak_rss_kernel_bytes` |
| `monitor.csv` | RSS gộp theo phút: `worker`, `run_id`, `minute` (epoch giây), `rss_max_bytes`, `rss_mean_bytes` |
| `disk.csv` | dung lượng `data/` theo thời gian: `worker`, `run_id`, `t`, `data_dir_bytes`, `scan_sec` |
"""

summary = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "config": run_config(),
    "data": DATA_STATS,
    "performance": PERF_STATS,
    "versions_distribution": versions_dist.to_dict(),
}
(EXPORT_DIR / "summary_stats.json").write_text(json.dumps(jsonable(summary), indent=2, ensure_ascii=False), encoding="utf-8")

export_cols = ["arxiv_id", "range_label", "metadata", "source", "refs", "num_versions", "n_versions_done",
               "n_versions_no_tex", "n_versions_not_found", "archive_bytes", "size_full_bytes", "size_tex_bib_bytes",
               "n_refs", "n_refs_arxiv", "time_metadata_sec", "time_source_sec", "time_refs_sec", "time_total_sec",
               "source_sleep_sec", "paper_success"]
per_paper[export_cols].to_csv(EXPORT_DIR / "per_paper.csv", index=False)
range_stats.to_csv(EXPORT_DIR / "range_stats.csv", index_label="range_label")
pd.DataFrame([(r, *RANGES[r]) for r in REPORT_RANGES],
             columns=["range_label", "yymm", "start", "end"]).to_csv(EXPORT_DIR / "ranges.csv", index=False)
category_counts.to_csv(EXPORT_DIR / "category_counts.csv")
runs_df.to_csv(EXPORT_DIR / "runs.csv", index=False)
if len(monitor_df):
    (monitor_df.assign(minute=(monitor_df["t"] // 60 * 60).astype("int64"))
     .groupby(["worker", "run_id", "minute"])["rss_bytes"].agg(rss_max_bytes="max", rss_mean_bytes="mean")
     .reset_index().to_csv(EXPORT_DIR / "monitor.csv", index=False))
disk_df.to_csv(EXPORT_DIR / "disk.csv", index=False)
(EXPORT_DIR / "README.md").write_text(EXPORT_README, encoding="utf-8")

export_size = dir_size(EXPORT_DIR)
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"{p.name:22s} {fmt_bytes(p.stat().st_size):>10}")
print(f"Tổng: {fmt_bytes(export_size)}" + ("  (CẢNH BÁO: > 10 MB)" if export_size > 10 * 2**20 else ""))

## 12. Kết luận

### 12.1 Tóm tắt kết quả

Bảng dưới gom các chỉ số chính của mục 9.5 và 10.1, cho phạm vi đang xét (`REPORT_ALL_RANGES`).

In [ ]:
# 12.1 — Tóm tắt các chỉ số chính
def pct(x):
    return "-" if x is None else f"{x:.1%}"


def num(x, digits=1):
    return "-" if x is None else f"{x:,.{digits}f}"


def secs(x):
    return "-" if x is None else fmt_secs(x)


D, P = DATA_STATS, PERF_STATS
key_results = pd.DataFrame([
    ("Số arXiv ID", f"{D['n_ids']:,}"),
    ("Bài cào thành công", f"{D['n_papers_success']:,} ({pct(D['success_rate_over_ids'])} số ID)"),
    ("Version đã xử lý (có TeX / chỉ PDF, khác / không tồn tại)",
     f"{D['n_versions_done']:,} / {D['n_versions_no_tex']:,} / {D['n_versions_not_found']:,}"),
    ("Kích thước TB mỗi bài: trước / sau khi bỏ hình",
     f"{fmt_bytes(D['avg_size_before_bytes'])} / {fmt_bytes(D['avg_size_after_bytes'])} "
     f"(giảm {pct(D['size_reduction_ratio'])})"),
    ("Reference TB mỗi bài (bài có reference)", num(D["avg_refs_per_paper_ok"])),
    ("Tỷ lệ lấy reference thành công", pct(D["refs_success_rate"])),
    ("Wall time (các máy song song) / thời gian máy", f"{secs(P['wall_time_sec'])} / {secs(P['machine_time_sec'])}"),
    ("Thời gian TB xử lý mỗi bài", f"{num(P['time_per_paper_mean_sec'])} s"),
    ("Tổng thời gian entry discovery", secs(P["entry_discovery_total_sec"])),
    ("RAM tối đa / trung bình", f"{fmt_bytes(P['rss_max_bytes'])} / {fmt_bytes(P['rss_mean_bytes'])}"),
    ("Disk tối đa / output cuối", f"{fmt_bytes(P['disk_max_bytes'])} / {fmt_bytes(P['output_size_bytes'])}"),
], columns=["Chỉ số", "Giá trị"]).set_index("Chỉ số")
display(key_results)

### 12.2 Hạn chế và hướng phát triển

- **Tốc độ bị giới hạn bởi quy tắc của arXiv**: mỗi máy chỉ một kết nối và ≥ 3 giây mỗi request, nên một máy cần khoảng 26 giờ cho 31.530 version. Nhóm chia range để chạy song song trên nhiều máy; mỗi máy vẫn đúng quy tắc.
- **429 của arXiv khó đoán**: export không trả header giới hạn tốc độ. Scraper tự giãn nhịp và nghỉ dài dần; số lần retry và cooldown được ghi lại (mục 10.1) để chỉnh `ARXIV_DELAY` / `COOLDOWNS_SEC`.
- **Semantic Scholar có thể chưa có bài mới** (độ trễ đánh chỉ mục khoảng 1–7 ngày), nên một số bài có kết quả `not_found`. Đặt `RETRY_NOT_FOUND = True` để hỏi lại các bài này ở phiên sau.
- **Version chỉ có PDF hoặc định dạng khác** không có TeX, nên chỉ có một thư mục rỗng như đề yêu cầu.
- **Hướng phát triển**: bản hybrid trong `src/ExportS3/` đọc v1 của các bài chỉ có một version từ bucket S3 bulk của arXiv. Theo ước tính, số request export giảm từ 31.530 xuống khoảng 11.700 (khoảng 10 giờ trên một máy). Bản này cần một tài khoản AWS đọc được bucket `s3://arxiv` (us-east-1).